# Leadership and Management Book Recommendation System

## Notebook 04 — Data Integration and Deduplication

### Purpose

This notebook integrates the cleaned Open Library Search API, Open Library Work API, and LeadershipNow datasets into a unified book-level analytical dataset.

The integration process prioritizes bibliographic identifiers and conservative matching rules while preserving source provenance and avoiding unsupported assumptions.

### Objectives

1. Reload and validate the cleaned source datasets.
2. Reconstruct serialized list-valued Open Library fields.
3. Integrate Open Library Search and Work API records using `openlibrary_key`.
4. Evaluate ISBN availability and overlap across data sources.
5. Resolve repeated LeadershipNow archive records at the canonical book level.
6. Match LeadershipNow and Open Library records using validated identifiers.
7. Apply conservative title-and-author matching only where reliable identifiers are unavailable.
8. Preserve unmatched records rather than forcing uncertain matches.
9. Document match methods and confidence levels.
10. Produce a unified dataset for SQL, EDA, statistics, NLP, clustering, and recommendation modeling.

## Matching and Manual Bibliographic Review

After exact normalized-title and author matching, unmatched LeadershipNow records were compared with Open Library titles using fuzzy string similarity.

Fuzzy matching was used only for **candidate generation** and was not treated as sufficient evidence of book identity.

One candidate required additional review:

- `Team Emotional Intelligence 2.0` — Jean Greaves and Evan Watkins
- `Emotional Intelligence 2.0` — Open Library record including Jean Greaves

Although the titles produced a high similarity score and shared an author, bibliographic review established that these are **distinct works**.

`Team Emotional Intelligence 2.0` was therefore rejected as a cross-source match and retained as an independent LeadershipNow book entity.

### Final Entity-Resolution Rule

Only records supported by exact normalized title and author evidence were accepted as confirmed cross-source matches.

Final results:

- Exact confirmed matches: **3**
- Accepted fuzzy matches: **0**
- Total confirmed cross-source matches: **3**

This demonstrates why fuzzy similarity is used only to identify records for review rather than automatically merging bibliographic entities.

In [1]:
import ast
import re
import numpy as np
import pandas as pd

from pathlib import Path

In [2]:
processed_dir = Path(
    "../data/processed"
)

final_dir = Path(
    "../data/final"
)

final_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
openlibrary_search = pd.read_csv(
    processed_dir /
    "open_library_search_clean.csv"
)

openlibrary_work = pd.read_csv(
    processed_dir /
    "open_library_work_enrichment_clean.csv"
)

leadershipnow = pd.read_csv(
    processed_dir /
    "leadershipnow_books_clean.csv"
)

In [4]:
print(
    "Open Library Search:",
    openlibrary_search.shape
)

print(
    "Open Library Work:",
    openlibrary_work.shape
)

print(
    "LeadershipNow:",
    leadershipnow.shape
)

Open Library Search: (950, 32)
Open Library Work: (950, 13)
LeadershipNow: (1124, 26)


In [5]:
source_key_audit = pd.DataFrame({
    "dataset": [
        "Open Library Search",
        "Open Library Work"
    ],

    "rows": [
        len(openlibrary_search),
        len(openlibrary_work)
    ],

    "unique_keys": [
        openlibrary_search[
            "openlibrary_key"
        ].nunique(),

        openlibrary_work[
            "openlibrary_key"
        ].nunique()
    ],

    "duplicate_keys": [
        openlibrary_search[
            "openlibrary_key"
        ].duplicated().sum(),

        openlibrary_work[
            "openlibrary_key"
        ].duplicated().sum()
    ],

    "missing_keys": [
        openlibrary_search[
            "openlibrary_key"
        ].isna().sum(),

        openlibrary_work[
            "openlibrary_key"
        ].isna().sum()
    ]
})

source_key_audit

,dataset,rows,unique_keys,duplicate_keys,missing_keys
0,Open Library Search,950,950,0,0
1,Open Library Work,950,950,0,0


In [6]:
search_keys = set(
    openlibrary_search[
        "openlibrary_key"
    ]
)

work_keys = set(
    openlibrary_work[
        "openlibrary_key"
    ]
)

print(
    "Search keys:",
    len(search_keys)
)

print(
    "Work keys:",
    len(work_keys)
)

print(
    "Search only:",
    len(
        search_keys - work_keys
    )
)

print(
    "Work only:",
    len(
        work_keys - search_keys
    )
)

print(
    "Shared keys:",
    len(
        search_keys & work_keys
    )
)

Search keys: 950
Work keys: 950
Search only: 0
Work only: 0
Shared keys: 950


In [7]:
shared_columns = sorted(
    set(openlibrary_search.columns)
    &
    set(openlibrary_work.columns)
)

shared_columns

['openlibrary_key', 'title']

In [8]:
title_check = (
    openlibrary_search[
        [
            "openlibrary_key",
            "title"
        ]
    ]
    .merge(
        openlibrary_work[
            [
                "openlibrary_key",
                "title"
            ]
        ],
        on="openlibrary_key",
        how="inner",
        suffixes=(
            "_search",
            "_work"
        )
    )
)

title_check[
    "title_match"
] = (
    title_check[
        "title_search"
    ]
    ==
    title_check[
        "title_work"
    ]
)

print(
    "Titles compared:",
    len(title_check)
)

print(
    "Matching titles:",
    title_check[
        "title_match"
    ].sum()
)

print(
    "Different titles:",
    (
        ~title_check[
            "title_match"
        ]
    ).sum()
)

Titles compared: 950
Matching titles: 950
Different titles: 0


In [9]:
openlibrary_work.columns.tolist()

['openlibrary_key',
 'title',
 'work_status_code',
 'description',
 'first_sentence',
 'work_subjects',
 'subject_places',
 'subject_people',
 'subject_times',
 'excerpts',
 'lc_classifications',
 'dewey_number',
 'work_covers']

In [10]:
openlibrary_work_merge = (
    openlibrary_work.copy()
)

In [11]:
openlibrary_work_merge = (
    openlibrary_work_merge.drop(
        columns=[
            "title"
        ],
        errors="ignore"
    )
)

In [12]:
openlibrary_integrated = (
    openlibrary_search.merge(
        openlibrary_work_merge,
        on="openlibrary_key",
        how="left",
        validate="one_to_one"
    )
)

In [13]:
validate="one_to_one"

In [14]:
print(
    "Integrated rows:",
    len(openlibrary_integrated)
)

print(
    "Integrated columns:",
    openlibrary_integrated.shape[1]
)

print(
    "Unique works:",
    openlibrary_integrated[
        "openlibrary_key"
    ].nunique()
)

print(
    "Duplicate work keys:",
    openlibrary_integrated[
        "openlibrary_key"
    ].duplicated().sum()
)

print(
    "Missing work keys:",
    openlibrary_integrated[
        "openlibrary_key"
    ].isna().sum()
)

Integrated rows: 950
Integrated columns: 43
Unique works: 950
Duplicate work keys: 0
Missing work keys: 0


In [15]:
work_fields = [
    "description",
    "first_sentence",
    "work_subjects",
    "subject_places",
    "subject_people",
    "subject_times",
    "excerpts",
    "lc_classifications",
    "dewey_number",
    "work_covers"
]

work_coverage = []

for column in work_fields:

    if column in openlibrary_integrated.columns:

        work_coverage.append({
            "field": column,
            "non_missing":
                openlibrary_integrated[
                    column
                ].notna().sum(),

            "missing":
                openlibrary_integrated[
                    column
                ].isna().sum(),

            "coverage_pct":
                round(
                    openlibrary_integrated[
                        column
                    ].notna().mean()
                    * 100,
                    2
                )
        })

work_coverage_df = pd.DataFrame(
    work_coverage
)

work_coverage_df

,field,non_missing,missing,coverage_pct
0,description,192,758,20.21
1,first_sentence,32,918,3.37
2,work_subjects,826,124,86.95
3,subject_places,56,894,5.89
4,subject_people,13,937,1.37
5,subject_times,9,941,0.95
6,excerpts,950,0,100.00
7,lc_classifications,85,865,8.95
8,dewey_number,155,795,16.32
9,work_covers,743,207,78.21


In [16]:
openlibrary_integrated[
    "source_openlibrary"
] = True

openlibrary_integrated[
    "source_leadershipnow"
] = False

In [17]:
openlibrary_integrated[
    "integration_source"
] = "Open Library"

## Open Library Integration

The cleaned Open Library Search and Work API datasets each contained 950 unique records with complete one-to-one alignment through `openlibrary_key`.

Before integration, key uniqueness and cross-dataset alignment were independently validated. All 950 Search API work identifiers were present in the Work API dataset, and no unmatched identifiers were identified.

The datasets were therefore merged using a one-to-one relationship on `openlibrary_key`. The Search API provides the primary bibliographic, identifier, rating, readership, and collection-query metadata, while the Work API contributes descriptive and subject-level enrichment.

Redundant title information was compared before integration rather than discarded without validation.

The resulting integrated Open Library dataset retains one record per Open Library work while combining the complementary metadata available from both API endpoints.

Source-provenance fields were added so that later cross-source integration with LeadershipNow can distinguish Open Library-only, LeadershipNow-only, and cross-source matched records.

In [18]:
def parse_serialized_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                return parsed

        except (ValueError, SyntaxError):
            pass

    return []

In [19]:
openlibrary_list_columns = [
    "authors",
    "author_keys",
    "publish_dates",
    "publishers",
    "isbn_10",
    "isbn_13",
    "all_isbns",
    "languages",
    "subjects",
    "collection_queries",
    "work_subjects",
    "subject_places",
    "subject_people",
    "subject_times",
    "excerpts",
    "lc_classifications",
    "dewey_number",
    "work_covers"
]

In [20]:
for column in openlibrary_list_columns:

    if column in openlibrary_integrated.columns:

        openlibrary_integrated[column] = (
            openlibrary_integrated[column]
            .apply(parse_serialized_list)
        )

In [21]:
for column in openlibrary_list_columns:

    if column in openlibrary_integrated.columns:

        valid_lists = (
            openlibrary_integrated[column]
            .apply(
                lambda x: isinstance(x, list)
            )
            .sum()
        )

        print(
            f"{column}: "
            f"{valid_lists}/{len(openlibrary_integrated)} lists"
        )

authors: 950/950 lists
author_keys: 950/950 lists
publish_dates: 950/950 lists
publishers: 950/950 lists
isbn_10: 950/950 lists
isbn_13: 950/950 lists
all_isbns: 950/950 lists
languages: 950/950 lists
subjects: 950/950 lists
collection_queries: 950/950 lists
work_subjects: 950/950 lists
subject_places: 950/950 lists
subject_people: 950/950 lists
subject_times: 950/950 lists
excerpts: 950/950 lists
lc_classifications: 950/950 lists
dewey_number: 950/950 lists
work_covers: 950/950 lists


In [22]:
work_list_fields = [
    "work_subjects",
    "subject_places",
    "subject_people",
    "subject_times",
    "excerpts",
    "lc_classifications",
    "dewey_number",
    "work_covers"
]

work_list_coverage = []

for column in work_list_fields:

    populated = (
        openlibrary_integrated[column]
        .apply(len)
        .gt(0)
        .sum()
    )

    work_list_coverage.append({
        "field": column,
        "populated_records": populated,
        "empty_records":
            len(openlibrary_integrated)
            - populated,
        "coverage_pct":
            round(
                populated
                / len(openlibrary_integrated)
                * 100,
                2
            )
    })

work_list_coverage_df = pd.DataFrame(
    work_list_coverage
)

work_list_coverage_df

,field,populated_records,empty_records,coverage_pct
0,work_subjects,826,124,86.95
1,subject_places,56,894,5.89
2,subject_people,13,937,1.37
3,subject_times,9,941,0.95
4,excerpts,23,927,2.42
5,lc_classifications,85,865,8.95
6,dewey_number,155,795,16.32
7,work_covers,743,207,78.21


In [23]:
openlibrary_integrated[
    "isbn13_count"
] = (
    openlibrary_integrated[
        "isbn_13"
    ].apply(len)
)

print(
    "Open Library works:",
    len(openlibrary_integrated)
)

print(
    "Works with ISBN-13:",
    (
        openlibrary_integrated[
            "isbn13_count"
        ] > 0
    ).sum()
)

print(
    "Works without ISBN-13:",
    (
        openlibrary_integrated[
            "isbn13_count"
        ] == 0
    ).sum()
)

print(
    "Total ISBN-13 entries:",
    openlibrary_integrated[
        "isbn13_count"
    ].sum()
)

print(
    "Maximum ISBN-13 values for one work:",
    openlibrary_integrated[
        "isbn13_count"
    ].max()
)

Open Library works: 950
Works with ISBN-13: 915
Works without ISBN-13: 35
Total ISBN-13 entries: 6048
Maximum ISBN-13 values for one work: 77


In [24]:
openlibrary_isbn_map = (
    openlibrary_integrated[
        [
            "openlibrary_key",
            "title",
            "authors",
            "isbn_13"
        ]
    ]
    .explode(
        "isbn_13"
    )
    .rename(
        columns={
            "isbn_13":
                "isbn_13_match"
        }
    )
)

In [25]:
openlibrary_isbn_map = (
    openlibrary_isbn_map.loc[
        openlibrary_isbn_map[
            "isbn_13_match"
        ].notna()
    ].copy()
)

In [26]:
openlibrary_isbn_map[
    "isbn_13_match"
] = (
    openlibrary_isbn_map[
        "isbn_13_match"
    ]
    .astype(str)
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
)

In [27]:
print(
    "ISBN mapping rows:",
    len(openlibrary_isbn_map)
)

print(
    "Unique ISBN-13 values:",
    openlibrary_isbn_map[
        "isbn_13_match"
    ].nunique()
)

print(
    "Open Library works represented:",
    openlibrary_isbn_map[
        "openlibrary_key"
    ].nunique()
)

ISBN mapping rows: 6048
Unique ISBN-13 values: 6012
Open Library works represented: 915


In [28]:
isbn_work_counts = (
    openlibrary_isbn_map
    .groupby(
        "isbn_13_match"
    )[
        "openlibrary_key"
    ]
    .nunique()
)

ambiguous_openlibrary_isbns = (
    isbn_work_counts[
        isbn_work_counts > 1
    ]
)

print(
    "ISBNs mapped to multiple Open Library works:",
    len(
        ambiguous_openlibrary_isbns
    )
)

ISBNs mapped to multiple Open Library works: 35


In [29]:
ambiguous_isbn_details = (
    openlibrary_isbn_map[
        openlibrary_isbn_map[
            "isbn_13_match"
        ].isin(
            ambiguous_openlibrary_isbns.index
        )
    ]
    .sort_values(
        [
            "isbn_13_match",
            "title"
        ]
    )
)

print(
    "Ambiguous ISBN values:",
    ambiguous_isbn_details[
        "isbn_13_match"
    ].nunique()
)

print(
    "Affected Open Library works:",
    ambiguous_isbn_details[
        "openlibrary_key"
    ].nunique()
)

print(
    "Affected mapping rows:",
    len(ambiguous_isbn_details)
)

ambiguous_isbn_details[
    [
        "isbn_13_match",
        "openlibrary_key",
        "title",
        "authors"
    ]
].head(50)

Ambiguous ISBN values: 35
Affected Open Library works: 48
Affected mapping rows: 70


,isbn_13_match,openlibrary_key,title,authors
666,9780071127301,/works/OL2714552W,Organizational behavior,"[John W. Newstrom, Keith Davis]"
696,9780071127301,/works/OL1267341W,Organizational behavior,[Keith Davis]
707,9780072487930,/works/OL3266746W,Managerial Economics & Business Strategy,"[Michael R. Baye, Michael Baye]"
748,9780072487930,/works/OL15843963W,Managerial economics and business strategy,[Michael R. Baye]
662,9780073530451,/works/OL547477W,Organizational behavior,"[Robert Kreitner, Angelo Kinicki]"
690,9780073530451,/works/OL18648685W,Organizational behavior,[Robert Kreitner]
492,9780130271471,/works/OL7942381W,Principles of operations management,[Jay Heizer]
501,9780130271471,/works/OL1887969W,Principles of operations management,[Jay H. Heizer]
297,9780130664921,/works/OL1982791W,Human Resource Management,[Gary Dessler]
621,9780130664921,/works/OL15643306W,Human resource management,[Gary Dessler]


In [30]:
ambiguous_distribution = (
    ambiguous_openlibrary_isbns
    .value_counts()
    .sort_index()
)

ambiguous_distribution

openlibrary_key
2    35
Name: count, dtype: int64

In [31]:
print(
    "Maximum works sharing one ISBN:",
    ambiguous_openlibrary_isbns.max()
)

Maximum works sharing one ISBN: 2


In [149]:
safe_openlibrary_isbns = (
    isbn_work_counts[
        isbn_work_counts == 1
    ].index
)

openlibrary_isbn_map_safe = (
    openlibrary_isbn_map[
        openlibrary_isbn_map[
            "isbn_13_match"
        ].isin(
            safe_openlibrary_isbns
        )
    ]
    .copy()
)

print(
    "All unique ISBN values:",
    openlibrary_isbn_map[
        "isbn_13_match"
    ].nunique()
)

print(
    "Safe unique ISBN values:",
    openlibrary_isbn_map_safe[
        "isbn_13_match"
    ].nunique()
)

print(
    "Excluded ambiguous ISBN values:",
    len(
        ambiguous_openlibrary_isbns
    )
)

print(
    "Open Library works represented "
    "by safe ISBNs:",
    openlibrary_isbn_map_safe[
        "openlibrary_key"
    ].nunique()
)

All unique ISBN values: 6012
Safe unique ISBN values: 5977
Excluded ambiguous ISBN values: 35
Open Library works represented by safe ISBNs: 906


In [150]:
leadershipnow[
    "isbn_status"
].value_counts(
    dropna=False
)

isbn_status
Valid ISBN-13          1061
Invalid check digit      57
Invalid length            3
Invalid characters        2
Invalid prefix            1
Name: count, dtype: int64

In [34]:
print(
    "LeadershipNow records:",
    len(leadershipnow)
)

print(
    "Validated ISBN-13 records:",
    leadershipnow[
        "isbn_13"
    ].notna().sum()
)

print(
    "Unique validated ISBN-13:",
    leadershipnow[
        "isbn_13"
    ].dropna().nunique()
)

LeadershipNow records: 1124
Validated ISBN-13 records: 1061
Unique validated ISBN-13: 1057


In [35]:
leadershipnow_isbn_map = (
    leadershipnow.loc[
        leadershipnow[
            "isbn_13"
        ].notna(),
        [
            "scrape_id",
            "title",
            "author",
            "isbn_13",
            "publication_year",
            "release_page"
        ]
    ]
    .copy()
)

In [36]:
leadershipnow_isbn_map[
    "isbn_13_match"
] = (
    leadershipnow_isbn_map[
        "isbn_13"
    ]
    .astype(str)
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
)

In [37]:
print(
    "LeadershipNow ISBN mapping rows:",
    len(leadershipnow_isbn_map)
)

print(
    "Unique LeadershipNow ISBNs:",
    leadershipnow_isbn_map[
        "isbn_13_match"
    ].nunique()
)

print(
    "Duplicate ISBN mapping rows:",
    leadershipnow_isbn_map[
        "isbn_13_match"
    ].duplicated(
        keep=False
    ).sum()
)

LeadershipNow ISBN mapping rows: 1061
Unique LeadershipNow ISBNs: 1057
Duplicate ISBN mapping rows: 8


In [38]:
openlibrary_safe_isbn_set = set(
    openlibrary_isbn_map_safe[
        "isbn_13_match"
    ]
)

leadershipnow_isbn_set = set(
    leadershipnow_isbn_map[
        "isbn_13_match"
    ]
)

exact_shared_isbns = (
    openlibrary_safe_isbn_set
    &
    leadershipnow_isbn_set
)

print(
    "Safe Open Library ISBNs:",
    len(openlibrary_safe_isbn_set)
)

print(
    "LeadershipNow validated ISBNs:",
    len(leadershipnow_isbn_set)
)

print(
    "Exact shared ISBNs:",
    len(exact_shared_isbns)
)

Safe Open Library ISBNs: 5977
LeadershipNow validated ISBNs: 1057
Exact shared ISBNs: 0


In [39]:
exact_isbn_matches = (
    leadershipnow_isbn_map[
        leadershipnow_isbn_map[
            "isbn_13_match"
        ].isin(
            exact_shared_isbns
        )
    ]
    .merge(
        openlibrary_isbn_map_safe[
            [
                "isbn_13_match",
                "openlibrary_key",
                "title",
                "authors"
            ]
        ],
        on="isbn_13_match",
        how="inner",
        suffixes=(
            "_leadershipnow",
            "_openlibrary"
        )
    )
)

In [40]:
print(
    "Exact ISBN match rows:",
    len(exact_isbn_matches)
)

print(
    "LeadershipNow records matched:",
    exact_isbn_matches[
        "scrape_id"
    ].nunique()
)

print(
    "Open Library works matched:",
    exact_isbn_matches[
        "openlibrary_key"
    ].nunique()
)

print(
    "Unique shared ISBNs:",
    exact_isbn_matches[
        "isbn_13_match"
    ].nunique()
)

Exact ISBN match rows: 0
LeadershipNow records matched: 0
Open Library works matched: 0
Unique shared ISBNs: 0


In [41]:
exact_isbn_matches[
    [
        "isbn_13_match",
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "publication_year"
    ]
].head(30)

,isbn_13_match,title_leadershipnow,author,title_openlibrary,authors,publication_year


## Cross-Source ISBN Matching Result

Exact ISBN-13 matching was evaluated as the highest-confidence method for linking LeadershipNow records to Open Library works.

Open Library contained 6,012 unique ISBN-13 values. After excluding 35 ISBN values that were associated with more than one Open Library work, 5,977 ISBN-13 values were considered safe for automatic identifier matching.

LeadershipNow contained 1,061 records with validated ISBN-13 values representing 1,057 unique ISBN-13 identifiers.

No exact ISBN-13 overlap was identified between the 5,977 safe Open Library identifiers and the 1,057 validated LeadershipNow identifiers.

This result does not demonstrate that the two datasets contain no overlapping books. ISBNs identify specific editions and formats, while Open Library organizes records primarily at the work level. A work may therefore appear in both datasets through different editions carrying different ISBN identifiers.

Consequently, ISBN matching alone cannot integrate the two sources. The next matching stage will use conservative bibliographic matching based on normalized titles and authors.

ISBN identifiers will remain preserved as source metadata and will not be modified or inferred to manufacture cross-source matches.

In [42]:
duplicate_leadershipnow_records = (
    leadershipnow[
        leadershipnow[
            "duplicate_isbn_group"
        ] == True
    ]
    .sort_values(
        [
            "isbn_normalized",
            "publication_date"
        ]
    )
)

duplicate_leadershipnow_records[
    [
        "title",
        "author",
        "isbn_normalized",
        "isbn_13",
        "publisher",
        "publication_date",
        "publication_year",
        "format",
        "page_count",
        "release_page"
    ]
]

,title,author,isbn_normalized,isbn_13,publisher,publication_date,publication_year,format,page_count,release_page
1101,Get It Done,Ayelet Fishbach,9780316538343,9.780317e+12,"Little, Brown Spark",2022-01-04,2022,Hardcover,304,2022 Releases
813,Get It Done,Ayelet Fishbach,9780316538343,9.780317e+12,"Little, Brown Spark",2023-01-04,2023,Hardcover,304,2023 Releases
1009,Burn Rate,Andy Dunn,9780593238264,9.780593e+12,Currency,2022-05-10,2022,Hardcover,304,2022 Releases
713,Burn Rate,Andy Dunn,9780593238264,9.780593e+12,Currency,2023-05-10,2023,Hardcover,304,2023 Releases
864,Win Every Argument,Mehdi Hasan,9781250853479,9.781251e+12,Henry Holt and Co.,2022-11-15,2022,Hardcover,240,2022 Releases
809,Win Every Argument,Mehdi Hasan,9781250853479,9.781251e+12,Henry Holt and Co.,2023-02-28,2023,Hardcover,336,2023 Releases
463,You're the Boss,Sabina Nawaz,9781668023181,9.781668e+12,Simon & Schuster,2024-03-04,2024,Hardcover,272,2024 Releases
181,You're the Boss,Sabina Nawaz,9781668023181,9.781668e+12,Simon & Schuster,2025-03-04,2025,Hardcover,272,2025 Releases


In [43]:
def normalize_title(value):

    if pd.isna(value):
        return ""

    value = str(value).lower()

    value = value.replace("’", "'")

    value = re.sub(
        r"[^a-z0-9\s]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [44]:
openlibrary_integrated[
    "title_normalized"
] = (
    openlibrary_integrated[
        "title"
    ].apply(normalize_title)
)

leadershipnow[
    "title_normalized"
] = (
    leadershipnow[
        "title"
    ].apply(normalize_title)
)

In [45]:
openlibrary_title_set = set(
    openlibrary_integrated.loc[
        openlibrary_integrated[
            "title_normalized"
        ] != "",
        "title_normalized"
    ]
)

leadershipnow_title_set = set(
    leadershipnow.loc[
        leadershipnow[
            "title_normalized"
        ] != "",
        "title_normalized"
    ]
)

shared_normalized_titles = (
    openlibrary_title_set
    &
    leadershipnow_title_set
)

print(
    "Unique Open Library titles:",
    len(openlibrary_title_set)
)

print(
    "Unique LeadershipNow titles:",
    len(leadershipnow_title_set)
)

print(
    "Shared normalized titles:",
    len(shared_normalized_titles)
)

Unique Open Library titles: 610
Unique LeadershipNow titles: 1113
Shared normalized titles: 4


In [46]:
title_match_candidates = (
    leadershipnow[
        leadershipnow[
            "title_normalized"
        ].isin(
            shared_normalized_titles
        )
    ][
        [
            "scrape_id",
            "title",
            "title_normalized",
            "author",
            "isbn_13",
            "publication_year"
        ]
    ]
    .merge(
        openlibrary_integrated[
            [
                "openlibrary_key",
                "title",
                "title_normalized",
                "authors",
                "first_publish_year"
            ]
        ],
        on="title_normalized",
        how="inner",
        suffixes=(
            "_leadershipnow",
            "_openlibrary"
        )
    )
)

In [47]:
print(
    "Candidate match rows:",
    len(title_match_candidates)
)

print(
    "LeadershipNow records involved:",
    title_match_candidates[
        "scrape_id"
    ].nunique()
)

print(
    "Open Library works involved:",
    title_match_candidates[
        "openlibrary_key"
    ].nunique()
)

Candidate match rows: 15
LeadershipNow records involved: 4
Open Library works involved: 15


In [48]:
title_match_candidates[
    [
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "publication_year",
        "first_publish_year"
    ]
].head(50)

,title_leadershipnow,author,title_openlibrary,authors,publication_year,first_publish_year
0,The Tao of Leadership,Jack Myers,The Tao of leadership,[John Heider],2025,1985.0
1,Emotional Intelligence Habits,Travis Bradberry,Emotional Intelligence Habits,[Travis Bradberry],2023,2023.0
2,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,The leadership challenge,"[James M. Kouzes, Barry Z. Posner, Elaine Biec...",2023,1987.0
3,Leadership,Henry Kissinger,Leadership,[Peter Guy Northouse],2022,1997.0
4,Leadership,Henry Kissinger,Leadership,[Peter G. Northouse],2022,2000.0
5,Leadership,Henry Kissinger,Leadership,[Elesa Zehndorfer],2022,2013.0
6,Leadership,Henry Kissinger,Leadership,"[Richard L. Hughes, Robert C. Ginnett, Gordon ...",2022,1993.0
7,Leadership,Henry Kissinger,Leadership,[James MacGregor Burns],2022,1978.0
8,Leadership,Henry Kissinger,Leadership,[Robert N. Lussier],2022,2000.0
9,Leadership,Henry Kissinger,Leadership,[Richard L. Hughes],2022,2012.0


In [49]:
def normalize_author_name(value):

    if pd.isna(value):
        return ""

    value = str(value).lower()

    value = value.replace("’", "'")

    value = re.sub(
        r"[^a-z0-9\s]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [50]:
openlibrary_integrated[
    "authors_normalized"
] = (
    openlibrary_integrated[
        "authors"
    ].apply(
        lambda authors: [
            normalize_author_name(author)
            for author in authors
            if normalize_author_name(author)
        ]
    )
)

In [51]:
leadershipnow[
    "author_normalized"
] = (
    leadershipnow[
        "author"
    ].apply(
        normalize_author_name
    )
)

In [52]:
title_match_candidates = (
    leadershipnow[
        leadershipnow[
            "title_normalized"
        ].isin(
            shared_normalized_titles
        )
    ][
        [
            "scrape_id",
            "title",
            "title_normalized",
            "author",
            "author_normalized",
            "isbn_13",
            "publication_year"
        ]
    ]
    .merge(
        openlibrary_integrated[
            [
                "openlibrary_key",
                "title",
                "title_normalized",
                "authors",
                "authors_normalized",
                "first_publish_year"
            ]
        ],
        on="title_normalized",
        how="inner",
        suffixes=(
            "_leadershipnow",
            "_openlibrary"
        )
    )
)

In [53]:
def find_author_matches(
    leadershipnow_author,
    openlibrary_authors
):

    matches = []

    for author in openlibrary_authors:

        if (
            author
            and author in leadershipnow_author
        ):
            matches.append(author)

    return matches

In [54]:
title_match_candidates[
    "matched_authors"
] = (
    title_match_candidates.apply(
        lambda row:
            find_author_matches(
                row[
                    "author_normalized"
                ],
                row[
                    "authors_normalized"
                ]
            ),
        axis=1
    )
)

In [55]:
title_match_candidates[
    "author_match_count"
] = (
    title_match_candidates[
        "matched_authors"
    ].apply(len)
)

title_match_candidates[
    "author_match"
] = (
    title_match_candidates[
        "author_match_count"
    ] > 0
)

In [56]:
title_match_candidates[
    [
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "matched_authors",
        "author_match_count",
        "author_match"
    ]
]

,title_leadershipnow,author,title_openlibrary,authors,matched_authors,author_match_count,author_match
0,The Tao of Leadership,Jack Myers,The Tao of leadership,[John Heider],[],0,False
1,Emotional Intelligence Habits,Travis Bradberry,Emotional Intelligence Habits,[Travis Bradberry],[travis bradberry],1,True
2,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,The leadership challenge,"[James M. Kouzes, Barry Z. Posner, Elaine Biec...","[james m kouzes, barry z posner]",2,True
3,Leadership,Henry Kissinger,Leadership,[Peter Guy Northouse],[],0,False
4,Leadership,Henry Kissinger,Leadership,[Peter G. Northouse],[],0,False
5,Leadership,Henry Kissinger,Leadership,[Elesa Zehndorfer],[],0,False
6,Leadership,Henry Kissinger,Leadership,"[Richard L. Hughes, Robert C. Ginnett, Gordon ...",[],0,False
7,Leadership,Henry Kissinger,Leadership,[James MacGregor Burns],[],0,False
8,Leadership,Henry Kissinger,Leadership,[Robert N. Lussier],[],0,False
9,Leadership,Henry Kissinger,Leadership,[Richard L. Hughes],[],0,False


In [57]:
high_confidence_matches = (
    title_match_candidates[
        title_match_candidates[
            "author_match"
        ]
    ]
    .copy()
)

In [58]:
print(
    "High-confidence match rows:",
    len(high_confidence_matches)
)

print(
    "LeadershipNow records matched:",
    high_confidence_matches[
        "scrape_id"
    ].nunique()
)

print(
    "Open Library works matched:",
    high_confidence_matches[
        "openlibrary_key"
    ].nunique()
)

High-confidence match rows: 3
LeadershipNow records matched: 3
Open Library works matched: 3


In [59]:
high_confidence_matches[
    [
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "matched_authors",
        "publication_year",
        "first_publish_year"
    ]
]

,title_leadershipnow,author,title_openlibrary,authors,matched_authors,publication_year,first_publish_year
1,Emotional Intelligence Habits,Travis Bradberry,Emotional Intelligence Habits,[Travis Bradberry],[travis bradberry],2023,2023.0
2,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,The leadership challenge,"[James M. Kouzes, Barry Z. Posner, Elaine Biec...","[james m kouzes, barry z posner]",2023,1987.0
13,Leadership,Henry Kissinger,Leadership,[Henry Kissinger],[henry kissinger],2022,2022.0


In [60]:
high_confidence_matches[
    "publication_year_difference"
] = (
    high_confidence_matches[
        "publication_year"
    ]
    -
    high_confidence_matches[
        "first_publish_year"
    ]
)

In [61]:
high_confidence_matches[
    [
        "title_leadershipnow",
        "author",
        "publication_year",
        "first_publish_year",
        "publication_year_difference"
    ]
]

,title_leadershipnow,author,publication_year,first_publish_year,publication_year_difference
1,Emotional Intelligence Habits,Travis Bradberry,2023,2023.0,0.0
2,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,2023,1987.0,36.0
13,Leadership,Henry Kissinger,2022,2022.0,0.0


In [62]:
title_match_candidates[
    "match_decision"
] = np.where(
    title_match_candidates[
        "author_match"
    ],
    "High-confidence work match",
    "Rejected - title only"
)

In [63]:
title_match_candidates[
    "match_decision"
].value_counts()

match_decision
Rejected - title only         12
High-confidence work match     3
Name: count, dtype: int64

## Exact Title and Author Matching

Because no exact cross-source ISBN-13 overlap was identified, work-level bibliographic matching was evaluated using normalized title and author information.

Only four normalized titles were shared between Open Library and LeadershipNow. These produced multiple candidate combinations because generic titles, particularly `Leadership`, corresponded to multiple distinct Open Library works.

Title equality alone was therefore not considered sufficient evidence of book identity.

Candidate pairs were evaluated using author evidence. A candidate was classified as a high-confidence work-level match when the normalized title matched exactly and at least one corresponding Open Library author was identified in the LeadershipNow author metadata.

Publication year was retained as supporting bibliographic evidence rather than requiring year equality because LeadershipNow may describe a recent edition of a work whose original publication occurred substantially earlier.

Candidates sharing a title but lacking author agreement were rejected rather than forcibly integrated.

This conservative approach prioritizes precision over match quantity and reduces the risk of incorrectly combining distinct books.

In [66]:
%pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [67]:
from rapidfuzz import fuzz, process

In [68]:
confirmed_leadershipnow_ids = set(
    high_confidence_matches[
        "scrape_id"
    ]
)

leadershipnow_unmatched = (
    leadershipnow[
        ~leadershipnow[
            "scrape_id"
        ].isin(
            confirmed_leadershipnow_ids
        )
    ]
    .copy()
)

print(
    "LeadershipNow records:",
    len(leadershipnow)
)

print(
    "Already confirmed:",
    len(
        confirmed_leadershipnow_ids
    )
)

print(
    "Remaining records:",
    len(
        leadershipnow_unmatched
    )
)

LeadershipNow records: 1124
Already confirmed: 3
Remaining records: 1121


In [69]:
openlibrary_title_lookup = (
    openlibrary_integrated[
        [
            "openlibrary_key",
            "title",
            "title_normalized",
            "authors",
            "authors_normalized",
            "first_publish_year"
        ]
    ]
    .copy()
)

In [70]:
openlibrary_title_choices = (
    openlibrary_title_lookup[
        "title_normalized"
    ]
    .tolist()
)

In [71]:
def get_best_title_match(title):

    if not title:
        return (
            None,
            None,
            None
        )

    result = process.extractOne(
        title,
        openlibrary_title_choices,
        scorer=fuzz.ratio
    )

    if result is None:
        return (
            None,
            None,
            None
        )

    matched_title, score, index = result

    return (
        matched_title,
        score,
        index
    )

In [72]:
fuzzy_results = (
    leadershipnow_unmatched[
        "title_normalized"
    ].apply(
        get_best_title_match
    )
)

leadershipnow_unmatched[
    "fuzzy_title_match"
] = (
    fuzzy_results.apply(
        lambda x: x[0]
    )
)

leadershipnow_unmatched[
    "title_similarity"
] = (
    fuzzy_results.apply(
        lambda x: x[1]
    )
)

leadershipnow_unmatched[
    "openlibrary_candidate_index"
] = (
    fuzzy_results.apply(
        lambda x: x[2]
    )
)

In [73]:
leadershipnow_unmatched[
    "title_similarity"
].describe()

count    1121.000000
mean       56.556914
std         8.853985
min        20.000000
25%        50.000000
50%        55.172414
75%        61.016949
max       100.000000
Name: title_similarity, dtype: float64

In [74]:
for threshold in [
    100,
    95,
    90,
    85,
    80,
    75
]:

    count = (
        leadershipnow_unmatched[
            "title_similarity"
        ] >= threshold
    ).sum()

    print(
        f">= {threshold}: {count}"
    )

>= 100: 1
>= 95: 1
>= 90: 3
>= 85: 7
>= 80: 21
>= 75: 53


In [75]:
strong_title_candidates = (
    leadershipnow_unmatched[
        leadershipnow_unmatched[
            "title_similarity"
        ] >= 85
    ]
    .sort_values(
        "title_similarity",
        ascending=False
    )
    .copy()
)

print(
    "Candidates with similarity >= 85:",
    len(strong_title_candidates)
)

Candidates with similarity >= 85: 7


In [76]:
strong_title_candidates[
    [
        "title",
        "author",
        "publication_year",
        "fuzzy_title_match",
        "title_similarity"
    ]
].head(50)

,title,author,publication_year,fuzzy_title_match,title_similarity
248,The Tao of Leadership,Jack Myers,2025,the tao of leadership,100.000000
1011,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,2022,emotional intelligence 2 0,91.228070
508,UnLeadership,Scott Stratten and Alison Stratten,2024,leadership,90.909091
385,The Soil of Leadership,Britt Yamamoto,2024,the tao of leadership,88.372093
323,On Leadership,Tony Blair,2024,leadership,86.956522
692,Meta-Leadership,Andrea Iorio,2023,team leadership,86.666667
146,The New Emotional Intelligence,Travis Bradberry,2025,the language of emotional intelligence,85.294118


In [77]:
fuzzy_candidate_details = (
    strong_title_candidates.merge(
        openlibrary_title_lookup[
            [
                "openlibrary_key",
                "title_normalized",
                "title",
                "authors",
                "authors_normalized",
                "first_publish_year"
            ]
        ],
        left_on="fuzzy_title_match",
        right_on="title_normalized",
        how="left",
        suffixes=(
            "_leadershipnow",
            "_openlibrary"
        )
    )
)

In [78]:
fuzzy_candidate_details[
    [
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "title_similarity",
        "publication_year",
        "first_publish_year"
    ]
].head(50)

,title_leadershipnow,author,title_openlibrary,authors,title_similarity,publication_year,first_publish_year
0,The Tao of Leadership,Jack Myers,The Tao of leadership,[John Heider],100.000000,2025,1985.0
1,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,Emotional Intelligence 2.0,"[Travis Bradberry, Jean Greaves, Jean Greaves ...",91.228070,2022,2009.0
2,UnLeadership,Scott Stratten and Alison Stratten,Leadership,[Peter Guy Northouse],90.909091,2024,1997.0
3,UnLeadership,Scott Stratten and Alison Stratten,Leadership,[Peter G. Northouse],90.909091,2024,2000.0
4,UnLeadership,Scott Stratten and Alison Stratten,Leadership,[Elesa Zehndorfer],90.909091,2024,2013.0
5,UnLeadership,Scott Stratten and Alison Stratten,Leadership,"[Richard L. Hughes, Robert C. Ginnett, Gordon ...",90.909091,2024,1993.0
6,UnLeadership,Scott Stratten and Alison Stratten,Leadership,[James MacGregor Burns],90.909091,2024,1978.0
7,UnLeadership,Scott Stratten and Alison Stratten,Leadership,[Robert N. Lussier],90.909091,2024,2000.0
8,UnLeadership,Scott Stratten and Alison Stratten,Leadership,[Richard L. Hughes],90.909091,2024,2012.0
9,UnLeadership,Scott Stratten and Alison Stratten,Leadership,"[Michael Z. Hackman, Craig E. Johnson]",90.909091,2024,1991.0


In [79]:
fuzzy_candidate_details[
    "matched_authors"
] = (
    fuzzy_candidate_details.apply(
        lambda row:
            find_author_matches(
                row[
                    "author_normalized"
                ],
                row[
                    "authors_normalized"
                ]
            ),
        axis=1
    )
)

In [80]:
fuzzy_candidate_details[
    "author_match_count"
] = (
    fuzzy_candidate_details[
        "matched_authors"
    ].apply(len)
)

fuzzy_candidate_details[
    "author_match"
] = (
    fuzzy_candidate_details[
        "author_match_count"
    ] > 0
)

In [81]:
fuzzy_author_supported = (
    fuzzy_candidate_details[
        fuzzy_candidate_details[
            "author_match"
        ]
    ]
    .sort_values(
        "title_similarity",
        ascending=False
    )
    .copy()
)

print(
    "Near-title candidates with author support:",
    len(fuzzy_author_supported)
)

Near-title candidates with author support: 1


In [82]:
fuzzy_author_supported[
    [
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "title_similarity",
        "matched_authors",
        "publication_year",
        "first_publish_year"
    ]
]

,title_leadershipnow,author,title_openlibrary,authors,title_similarity,matched_authors,publication_year,first_publish_year
1,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,Emotional Intelligence 2.0,"[Travis Bradberry, Jean Greaves, Jean Greaves ...",91.22807,[jean greaves],2022,2009.0


In [83]:
confirmed_fuzzy_matches = (
    fuzzy_author_supported[
        fuzzy_author_supported[
            "title_similarity"
        ] >= 90
    ]
    .copy()
)

print(
    "Confirmed fuzzy matches:",
    len(confirmed_fuzzy_matches)
)

Confirmed fuzzy matches: 1


In [84]:
confirmed_fuzzy_matches[
    [
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "title_similarity",
        "matched_authors",
        "publication_year",
        "first_publish_year"
    ]
]

,title_leadershipnow,author,title_openlibrary,authors,title_similarity,matched_authors,publication_year,first_publish_year
1,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,Emotional Intelligence 2.0,"[Travis Bradberry, Jean Greaves, Jean Greaves ...",91.22807,[jean greaves],2022,2009.0


In [85]:
high_confidence_matches[
    "match_method"
] = "Exact normalized title + author"

In [86]:
confirmed_fuzzy_matches[
    "match_method"
] = "Near title >= 90 + author"

In [87]:
high_confidence_matches[
    "match_confidence"
] = "High"

confirmed_fuzzy_matches[
    "match_confidence"
] = "High"

In [88]:
exact_confirmed = (
    high_confidence_matches[
        [
            "scrape_id",
            "openlibrary_key",
            "title_leadershipnow",
            "author",
            "title_openlibrary",
            "authors",
            "publication_year",
            "first_publish_year",
            "match_method",
            "match_confidence"
        ]
    ]
    .copy()
)

In [89]:
fuzzy_confirmed = (
    confirmed_fuzzy_matches[
        [
            "scrape_id",
            "openlibrary_key",
            "title_leadershipnow",
            "author",
            "title_openlibrary",
            "authors",
            "publication_year",
            "first_publish_year",
            "match_method",
            "match_confidence"
        ]
    ]
    .copy()
)

In [90]:
confirmed_cross_source_matches = (
    pd.concat(
        [
            exact_confirmed,
            fuzzy_confirmed
        ],
        ignore_index=True
    )
)

In [91]:
print(
    "Confirmed match rows:",
    len(
        confirmed_cross_source_matches
    )
)

print(
    "Unique LeadershipNow records:",
    confirmed_cross_source_matches[
        "scrape_id"
    ].nunique()
)

print(
    "Unique Open Library works:",
    confirmed_cross_source_matches[
        "openlibrary_key"
    ].nunique()
)

print(
    "\nMatching methods:"
)

print(
    confirmed_cross_source_matches[
        "match_method"
    ].value_counts()
)

Confirmed match rows: 4
Unique LeadershipNow records: 4
Unique Open Library works: 4

Matching methods:
match_method
Exact normalized title + author    3
Near title >= 90 + author          1
Name: count, dtype: int64


In [92]:
confirmed_cross_source_matches[
    [
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "publication_year",
        "first_publish_year",
        "match_method",
        "match_confidence"
    ]
]

,title_leadershipnow,author,title_openlibrary,authors,publication_year,first_publish_year,match_method,match_confidence
0,Emotional Intelligence Habits,Travis Bradberry,Emotional Intelligence Habits,[Travis Bradberry],2023,2023.0,Exact normalized title + author,High
1,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,The leadership challenge,"[James M. Kouzes, Barry Z. Posner, Elaine Biec...",2023,1987.0,Exact normalized title + author,High
2,Leadership,Henry Kissinger,Leadership,[Henry Kissinger],2022,2022.0,Exact normalized title + author,High
3,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,Emotional Intelligence 2.0,"[Travis Bradberry, Jean Greaves, Jean Greaves ...",2022,2009.0,Near title >= 90 + author,High


In [93]:
print(
    "Duplicate LeadershipNow IDs:",
    confirmed_cross_source_matches[
        "scrape_id"
    ].duplicated().sum()
)

print(
    "Duplicate Open Library keys:",
    confirmed_cross_source_matches[
        "openlibrary_key"
    ].duplicated().sum()
)

Duplicate LeadershipNow IDs: 0
Duplicate Open Library keys: 0


## Near-Title Matching

After exact normalized-title and author matching, unmatched LeadershipNow records were compared with Open Library titles using fuzzy string similarity.

Fuzzy title similarity was used only for candidate generation and was not treated as sufficient evidence of book identity.

The similarity distribution demonstrated the risk of relying on title similarity alone. Several unrelated books containing the word `Leadership` produced high similarity scores despite having different authors and representing different works.

Only one near-title candidate also contained supporting author evidence:

- `Team Emotional Intelligence 2.0` by Jean Greaves and Evan Watkins
- `Emotional Intelligence 2.0` in Open Library, whose author metadata includes Jean Greaves

The titles produced a similarity score of approximately 91.23%.

This candidate was retained as a high-confidence work-level match because both title similarity and independent author evidence supported the relationship.

A 90% title-similarity threshold was used only within this author-supported candidate set and should not be interpreted as a universal bibliographic matching threshold.

All other fuzzy-title candidates lacking author evidence were rejected rather than forcibly merged.

In [94]:
matching_summary = pd.DataFrame({
    "matching_stage": [
        "Exact validated ISBN-13",
        "Exact normalized title + author",
        "Near title >= 90 + author"
    ],

    "confirmed_matches": [
        0,
        len(high_confidence_matches),
        len(confirmed_fuzzy_matches)
    ]
})

matching_summary

,matching_stage,confirmed_matches
0,Exact validated ISBN-13,0
1,Exact normalized title + author,3
2,Near title >= 90 + author,1


In [95]:
duplicate_leadershipnow_records = (
    leadershipnow[
        leadershipnow[
            "duplicate_isbn_group"
        ] == True
    ]
    .sort_values(
        [
            "isbn_normalized",
            "publication_date"
        ]
    )
)

duplicate_leadershipnow_records[
    [
        "scrape_id",
        "title",
        "author",
        "isbn_normalized",
        "isbn_13",
        "publisher",
        "publication_date",
        "publication_year",
        "format",
        "page_count",
        "release_page"
    ]
]

,scrape_id,title,author,isbn_normalized,isbn_13,publisher,publication_date,publication_year,format,page_count,release_page
1101,1102,Get It Done,Ayelet Fishbach,9780316538343,9.780317e+12,"Little, Brown Spark",2022-01-04,2022,Hardcover,304,2022 Releases
813,814,Get It Done,Ayelet Fishbach,9780316538343,9.780317e+12,"Little, Brown Spark",2023-01-04,2023,Hardcover,304,2023 Releases
1009,1010,Burn Rate,Andy Dunn,9780593238264,9.780593e+12,Currency,2022-05-10,2022,Hardcover,304,2022 Releases
713,714,Burn Rate,Andy Dunn,9780593238264,9.780593e+12,Currency,2023-05-10,2023,Hardcover,304,2023 Releases
864,865,Win Every Argument,Mehdi Hasan,9781250853479,9.781251e+12,Henry Holt and Co.,2022-11-15,2022,Hardcover,240,2022 Releases
809,810,Win Every Argument,Mehdi Hasan,9781250853479,9.781251e+12,Henry Holt and Co.,2023-02-28,2023,Hardcover,336,2023 Releases
463,464,You're the Boss,Sabina Nawaz,9781668023181,9.781668e+12,Simon & Schuster,2024-03-04,2024,Hardcover,272,2024 Releases
181,182,You're the Boss,Sabina Nawaz,9781668023181,9.781668e+12,Simon & Schuster,2025-03-04,2025,Hardcover,272,2025 Releases


In [96]:
duplicate_consistency = (
    duplicate_leadershipnow_records
    .groupby(
        "isbn_normalized"
    )
    .agg(
        records=("scrape_id", "size"),
        unique_titles=("title", "nunique"),
        unique_authors=("author", "nunique"),
        unique_publishers=("publisher", "nunique"),
        unique_formats=("format", "nunique"),
        unique_page_counts=("page_count", "nunique"),
        unique_publication_dates=(
            "publication_date",
            "nunique"
        ),
        unique_release_pages=(
            "release_page",
            "nunique"
        )
    )
    .reset_index()
)

duplicate_consistency

,isbn_normalized,records,unique_titles,unique_authors,unique_publishers,unique_formats,unique_page_counts,unique_publication_dates,unique_release_pages
0,9780316538343,2,1,1,1,1,1,2,2
1,9780593238264,2,1,1,1,1,1,2,2
2,9781250853479,2,1,1,1,1,2,2,2
3,9781668023181,2,1,1,1,1,1,2,2


In [97]:
canonical_date_map = (
    duplicate_leadershipnow_records
    .groupby(
        "isbn_normalized"
    )[
        "publication_date"
    ]
    .min()
)

canonical_date_map

isbn_normalized
9780316538343    2022-01-04
9780593238264    2022-05-10
9781250853479    2022-11-15
9781668023181    2024-03-04
Name: publication_date, dtype: str

In [98]:
page_count_conflicts = (
    duplicate_consistency[
        duplicate_consistency[
            "unique_page_counts"
        ] > 1
    ]
)

page_count_conflicts

,isbn_normalized,records,unique_titles,unique_authors,unique_publishers,unique_formats,unique_page_counts,unique_publication_dates,unique_release_pages
2,9781250853479,2,1,1,1,1,2,2,2


In [99]:
conflicting_page_records = (
    duplicate_leadershipnow_records[
        duplicate_leadershipnow_records[
            "isbn_normalized"
        ].isin(
            page_count_conflicts[
                "isbn_normalized"
            ]
        )
    ]
)

conflicting_page_records[
    [
        "title",
        "isbn_normalized",
        "publication_date",
        "page_count",
        "release_page"
    ]
]

,title,isbn_normalized,publication_date,page_count,release_page
864,Win Every Argument,9781250853479,2022-11-15,240,2022 Releases
809,Win Every Argument,9781250853479,2023-02-28,336,2023 Releases


In [100]:
archive_provenance = (
    leadershipnow
    .groupby(
        "isbn_normalized",
        dropna=False
    )[
        "release_page"
    ]
    .agg(
        lambda x: list(
            dict.fromkeys(
                value
                for value in x
                if pd.notna(value)
            )
        )
    )
    .reset_index(
        name="release_pages"
    )
)

In [101]:
archive_provenance[
    archive_provenance[
        "isbn_normalized"
    ].isin(
        duplicate_leadershipnow_records[
            "isbn_normalized"
        ]
    )
]

,isbn_normalized,release_pages
111,9780316538343,"[2023 Releases, 2022 Releases]"
147,9780593238264,"[2023 Releases, 2022 Releases]"
335,9781250853479,"[2023 Releases, 2022 Releases]"
853,9781668023181,"[2025 Releases, 2024 Releases]"


In [102]:
leadershipnow_with_isbn = (
    leadershipnow[
        leadershipnow[
            "isbn_normalized"
        ].notna()
    ]
    .copy()
)

leadershipnow_without_isbn = (
    leadershipnow[
        leadershipnow[
            "isbn_normalized"
        ].isna()
    ]
    .copy()
)

print(
    "With source ISBN:",
    len(leadershipnow_with_isbn)
)

print(
    "Without source ISBN:",
    len(leadershipnow_without_isbn)
)

With source ISBN: 1124
Without source ISBN: 0


In [103]:
leadershipnow_with_isbn = (
    leadershipnow_with_isbn
    .sort_values(
        [
            "isbn_normalized",
            "publication_date"
        ]
    )
)

leadershipnow_canonical_isbn = (
    leadershipnow_with_isbn
    .drop_duplicates(
        subset=[
            "isbn_normalized"
        ],
        keep="first"
    )
    .copy()
)

In [104]:
leadershipnow_canonical_isbn = (
    leadershipnow_canonical_isbn
    .merge(
        archive_provenance,
        on="isbn_normalized",
        how="left"
    )
)

In [105]:
conflict_isbns = set(
    duplicate_consistency.loc[
        (
            duplicate_consistency[
                "unique_titles"
            ] > 1
        )
        |
        (
            duplicate_consistency[
                "unique_authors"
            ] > 1
        )
        |
        (
            duplicate_consistency[
                "unique_publishers"
            ] > 1
        )
        |
        (
            duplicate_consistency[
                "unique_formats"
            ] > 1
        )
        |
        (
            duplicate_consistency[
                "unique_page_counts"
            ] > 1
        ),
        "isbn_normalized"
    ]
)

In [106]:
leadershipnow_canonical_isbn[
    "source_metadata_conflict"
] = (
    leadershipnow_canonical_isbn[
        "isbn_normalized"
    ].isin(
        conflict_isbns
    )
)

In [107]:
page_conflict_isbns = set(
    page_count_conflicts[
        "isbn_normalized"
    ]
)

leadershipnow_canonical_isbn.loc[
    leadershipnow_canonical_isbn[
        "isbn_normalized"
    ].isin(
        page_conflict_isbns
    ),
    "page_count"
] = pd.NA

In [108]:
leadershipnow_canonical = (
    pd.concat(
        [
            leadershipnow_canonical_isbn,
            leadershipnow_without_isbn
        ],
        ignore_index=True
    )
)

In [109]:
print(
    "Original LeadershipNow source records:",
    len(leadershipnow)
)

print(
    "Canonical LeadershipNow records:",
    len(leadershipnow_canonical)
)

print(
    "Records consolidated:",
    len(leadershipnow)
    -
    len(leadershipnow_canonical)
)

print(
    "Unique normalized ISBNs:",
    leadershipnow_canonical[
        "isbn_normalized"
    ].dropna().nunique()
)

print(
    "Metadata conflict records:",
    leadershipnow_canonical[
        "source_metadata_conflict"
    ].fillna(False).sum()
)

Original LeadershipNow source records: 1124
Canonical LeadershipNow records: 1120
Records consolidated: 4
Unique normalized ISBNs: 1120
Metadata conflict records: 1


## LeadershipNow Canonicalization

LeadershipNow contained repeated archive records in which the same source ISBN appeared across more than one annual release archive.

These records were not removed during source cleaning because their repeated archive appearances represent source provenance. Canonicalization was therefore performed only during the integration stage.

Records sharing the same normalized source ISBN were consolidated into a single canonical LeadershipNow record while preserving the complete list of release archives in which the source record appeared.

The earliest publication date observed in LeadershipNow was retained as the canonical source date. This value represents the earliest date observed within the collected LeadershipNow archives and is not interpreted as independently verified global publication history.

Where repeated source records contained conflicting bibliographic metadata, the conflict was explicitly flagged. Conflicting page-count values were not resolved arbitrarily; the canonical page-count value was treated as missing unless supported by consistent evidence.

This approach reduces repeated archive records without discarding provenance or converting source disagreement into unsupported certainty.

In [110]:
confirmed_match_titles = set(
    confirmed_cross_source_matches[
        "title_leadershipnow"
    ].apply(normalize_title)
)

canonical_confirmed_check = (
    leadershipnow_canonical[
        leadershipnow_canonical[
            "title_normalized"
        ].isin(
            confirmed_match_titles
        )
    ][
        [
            "scrape_id",
            "title",
            "author",
            "isbn_normalized",
            "publication_year"
        ]
    ]
)

canonical_confirmed_check

,scrape_id,title,author,isbn_normalized,publication_year
178,956,Leadership,Henry Kissinger,9780593489444,2022
239,1012,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,9780974719344,2022
240,657,Emotional Intelligence Habits,Travis Bradberry,9780974719375,2023
261,815,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,9781119736127,2023


In [111]:
print(
    "Expected confirmed works:",
    len(confirmed_cross_source_matches)
)

print(
    "Canonical LeadershipNow records found:",
    len(canonical_confirmed_check)
)

Expected confirmed works: 4
Canonical LeadershipNow records found: 4


In [112]:
confirmed_match_map = (
    confirmed_cross_source_matches[
        [
            "openlibrary_key",
            "title_leadershipnow",
            "author",
            "match_method",
            "match_confidence"
        ]
    ]
    .copy()
)

confirmed_match_map[
    "leadershipnow_title_normalized"
] = (
    confirmed_match_map[
        "title_leadershipnow"
    ].apply(normalize_title)
)

confirmed_match_map[
    "leadershipnow_author_normalized"
] = (
    confirmed_match_map[
        "author"
    ].apply(normalize_author_name)
)

In [113]:
confirmed_match_map

,openlibrary_key,title_leadershipnow,author,match_method,match_confidence,leadershipnow_title_normalized,leadershipnow_author_normalized
0,/works/OL35744456W,Emotional Intelligence Habits,Travis Bradberry,Exact normalized title + author,High,emotional intelligence habits,travis bradberry
1,/works/OL1974749W,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,Exact normalized title + author,High,the leadership challenge,james m kouzes and barry z posner
2,/works/OL25348188W,Leadership,Henry Kissinger,Exact normalized title + author,High,leadership,henry kissinger
3,/works/OL15189300W,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,Near title >= 90 + author,High,team emotional intelligence 2 0,jean greaves and evan watkins


In [114]:
leadershipnow_canonical[
    "title_normalized"
] = (
    leadershipnow_canonical[
        "title"
    ].apply(normalize_title)
)

leadershipnow_canonical[
    "author_normalized"
] = (
    leadershipnow_canonical[
        "author"
    ].apply(normalize_author_name)
)

In [115]:
leadershipnow_canonical = (
    leadershipnow_canonical.merge(
        confirmed_match_map[
            [
                "openlibrary_key",
                "leadershipnow_title_normalized",
                "leadershipnow_author_normalized",
                "match_method",
                "match_confidence"
            ]
        ],
        left_on=[
            "title_normalized",
            "author_normalized"
        ],
        right_on=[
            "leadershipnow_title_normalized",
            "leadershipnow_author_normalized"
        ],
        how="left",
        validate="many_to_one"
    )
)

In [116]:
print(
    "Canonical LeadershipNow records:",
    len(leadershipnow_canonical)
)

print(
    "Cross-source matched records:",
    leadershipnow_canonical[
        "openlibrary_key"
    ].notna().sum()
)

Canonical LeadershipNow records: 1120
Cross-source matched records: 4


In [117]:
leadershipnow_matched = (
    leadershipnow_canonical[
        leadershipnow_canonical[
            "openlibrary_key"
        ].notna()
    ]
    .copy()
)

leadershipnow_unmatched_canonical = (
    leadershipnow_canonical[
        leadershipnow_canonical[
            "openlibrary_key"
        ].isna()
    ]
    .copy()
)

In [118]:
print(
    "Matched LeadershipNow books:",
    len(leadershipnow_matched)
)

print(
    "Unmatched LeadershipNow books:",
    len(leadershipnow_unmatched_canonical)
)

print(
    "Total:",
    len(leadershipnow_matched)
    +
    len(leadershipnow_unmatched_canonical)
)

Matched LeadershipNow books: 4
Unmatched LeadershipNow books: 1116
Total: 1120


In [119]:
openlibrary_work_count = (
    openlibrary_integrated[
        "openlibrary_key"
    ].nunique()
)

leadershipnow_canonical_count = (
    len(
        leadershipnow_canonical
    )
)

cross_source_match_count = (
    leadershipnow_canonical[
        "openlibrary_key"
    ].notna().sum()
)

expected_unified_books = (
    openlibrary_work_count
    +
    leadershipnow_canonical_count
    -
    cross_source_match_count
)

print(
    "Open Library works:",
    openlibrary_work_count
)

print(
    "Canonical LeadershipNow books:",
    leadershipnow_canonical_count
)

print(
    "Confirmed cross-source matches:",
    cross_source_match_count
)

print(
    "Expected unified books:",
    expected_unified_books
)

Open Library works: 950
Canonical LeadershipNow books: 1120
Confirmed cross-source matches: 4
Expected unified books: 2066


In [120]:
unified_book_ids = {}
book_counter = 1

for openlibrary_key in (
    openlibrary_integrated[
        "openlibrary_key"
    ]
):

    unified_book_ids[
        ("openlibrary", openlibrary_key)
    ] = f"BOOK{book_counter:05d}"

    book_counter += 1

In [121]:
leadershipnow_book_ids = {}

for _, row in leadershipnow_canonical.iterrows():

    if pd.notna(
        row["openlibrary_key"]
    ):

        book_id = unified_book_ids[
            (
                "openlibrary",
                row["openlibrary_key"]
            )
        ]

    else:

        book_id = f"BOOK{book_counter:05d}"
        book_counter += 1

    leadershipnow_book_ids[
        row["scrape_id"]
    ] = book_id

In [122]:
openlibrary_integrated[
    "book_id"
] = (
    openlibrary_integrated[
        "openlibrary_key"
    ].apply(
        lambda key:
            unified_book_ids[
                ("openlibrary", key)
            ]
    )
)

In [123]:
leadershipnow_canonical[
    "book_id"
] = (
    leadershipnow_canonical[
        "scrape_id"
    ].map(
        leadershipnow_book_ids
    )
)

In [124]:
print(
    "Open Library rows:",
    len(openlibrary_integrated)
)

print(
    "Open Library unique book IDs:",
    openlibrary_integrated[
        "book_id"
    ].nunique()
)

print(
    "LeadershipNow rows:",
    len(leadershipnow_canonical)
)

print(
    "LeadershipNow unique book IDs:",
    leadershipnow_canonical[
        "book_id"
    ].nunique()
)

Open Library rows: 950
Open Library unique book IDs: 950
LeadershipNow rows: 1120
LeadershipNow unique book IDs: 1120


In [125]:
shared_book_ids = (
    set(
        openlibrary_integrated[
            "book_id"
        ]
    )
    &
    set(
        leadershipnow_canonical[
            "book_id"
        ]
    )
)

print(
    "Shared cross-source book IDs:",
    len(shared_book_ids)
)

Shared cross-source book IDs: 4


In [126]:
all_book_ids = (
    set(
        openlibrary_integrated[
            "book_id"
        ]
    )
    |
    set(
        leadershipnow_canonical[
            "book_id"
        ]
    )
)

print(
    "Total unique unified book IDs:",
    len(all_book_ids)
)

Total unique unified book IDs: 2066


## Unified Book Entity Resolution

Following source-level cleaning and canonicalization, the Open Library dataset contained 950 unique works and LeadershipNow contained 1,120 canonical book records.

Cross-source entity resolution identified four high-confidence work-level relationships. These relationships were consolidated at the project identifier level rather than deleting either source record.

Each unique book entity was assigned a project-specific `book_id`. When Open Library and LeadershipNow records were determined to represent the same underlying work, both source records were assigned the same project `book_id`. Source-specific identifiers, metadata, and provenance remained preserved.

This architecture separates book identity from source representation and allows conflicting or complementary metadata from multiple sources to coexist without losing provenance.

In [127]:
master_columns = [
    "book_id",
    "title",
    "authors",
    "description",
    "categories",
    "publisher",
    "published_date",
    "publication_year",
    "first_publish_year",
    "isbn_10",
    "isbn_13",
    "language",
    "page_count",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "format",
    "cover_url",
    "book_url",
    "openlibrary_key",
    "source_openlibrary",
    "source_leadershipnow",
    "source_count",
    "match_method",
    "match_confidence"
]

In [148]:
openlibrary_integrated.columns.tolist()

['openlibrary_key',
 'title',
 'authors',
 'author_keys',
 'first_publish_year',
 'publish_dates',
 'publishers',
 'isbn_10',
 'isbn_13',
 'all_isbns',
 'languages',
 'subjects',
 'edition_count',
 'ratings_average',
 'ratings_count',
 'ratings_count_1',
 'ratings_count_2',
 'ratings_count_3',
 'ratings_count_4',
 'ratings_count_5',
 'want_to_read_count',
 'currently_reading_count',
 'already_read_count',
 'cover_id',
 'cover_url',
 'ebook_access',
 'has_fulltext',
 'public_scan',
 'source',
 'source_url',
 'collection_queries',
 'query_match_count',
 'work_status_code',
 'description',
 'first_sentence',
 'work_subjects',
 'subject_places',
 'subject_people',
 'subject_times',
 'excerpts',
 'lc_classifications',
 'dewey_number',
 'work_covers',
 'source_openlibrary',
 'source_leadershipnow',
 'integration_source',
 'isbn13_count',
 'title_normalized',
 'authors_normalized',
 'book_id']

In [131]:
[
    column
    for column in openlibrary_integrated.columns
    if "url" in column.lower()
    or "link" in column.lower()
    or "key" in column.lower()
]

['openlibrary_key', 'author_keys', 'cover_url', 'source_url']

In [132]:
leadershipnow_canonical.columns.tolist()

['scrape_id',
 'title',
 'subtitle',
 'author',
 'format_raw',
 'isbn',
 'publisher',
 'publication_date_raw',
 'cover_url',
 'cover_alt_raw',
 'external_book_url',
 'release_page',
 'release_page_url',
 'source',
 'scraped_at',
 'isbn_normalized',
 'isbn13_valid',
 'format',
 'page_count',
 'publication_date',
 'publication_year',
 'edition_note',
 'isbn_13',
 'isbn_status',
 'duplicate_isbn_group',
 'duplicate_title',
 'title_normalized',
 'author_normalized',
 'release_pages',
 'source_metadata_conflict',
 'openlibrary_key',
 'leadershipnow_title_normalized',
 'leadershipnow_author_normalized',
 'match_method',
 'match_confidence',
 'book_id']

In [133]:
openlibrary_master = pd.DataFrame({
    "book_id":
        openlibrary_integrated["book_id"],

    "title":
        openlibrary_integrated["title"],

    "authors":
        openlibrary_integrated["authors"],

    "description":
        openlibrary_integrated["description"],

    "categories":
        openlibrary_integrated["work_subjects"],

    "publishers":
        openlibrary_integrated["publishers"],

    "publish_dates":
        openlibrary_integrated["publish_dates"],

    "first_publish_year":
        openlibrary_integrated["first_publish_year"],

    "isbn_10":
        openlibrary_integrated["isbn_10"],

    "isbn_13":
        openlibrary_integrated["isbn_13"],

    "languages":
        openlibrary_integrated["languages"],

    "average_rating":
        openlibrary_integrated["ratings_average"],

    "ratings_count":
        openlibrary_integrated["ratings_count"],

    "edition_count":
        openlibrary_integrated["edition_count"],

    "want_to_read_count":
        openlibrary_integrated["want_to_read_count"],

    "currently_reading_count":
        openlibrary_integrated["currently_reading_count"],

    "already_read_count":
        openlibrary_integrated["already_read_count"],

    "cover_url":
        openlibrary_integrated["cover_url"],

    "book_url":
        openlibrary_integrated["source_url"],

    "openlibrary_key":
        openlibrary_integrated["openlibrary_key"],

    "collection_queries":
        openlibrary_integrated["collection_queries"],

    "query_match_count":
        openlibrary_integrated["query_match_count"],

    "source":
        "Open Library"
})

In [135]:
matched_openlibrary_records = (
    openlibrary_integrated[
        openlibrary_integrated[
            "book_id"
        ].isin(
            shared_book_ids
        )
    ]
    .copy()
)

matched_leadershipnow_records = (
    leadershipnow_canonical[
        leadershipnow_canonical[
            "book_id"
        ].isin(
            shared_book_ids
        )
    ]
    .copy()
)

In [136]:
print(
    "Matched Open Library rows:",
    len(matched_openlibrary_records)
)

print(
    "Matched LeadershipNow rows:",
    len(matched_leadershipnow_records)
)

Matched Open Library rows: 4
Matched LeadershipNow rows: 4


In [137]:
matched_book_comparison = (
    matched_leadershipnow_records[
        [
            "book_id",
            "title",
            "author",
            "publisher",
            "publication_date",
            "publication_year",
            "isbn_13",
            "page_count"
        ]
    ]
    .merge(
        matched_openlibrary_records[
            [
                "book_id",
                "title",
                "authors",
                "publishers",
                "first_publish_year",
                "isbn_13",
                "description",
                "ratings_average",
                "ratings_count"
            ]
        ],
        on="book_id",
        how="inner",
        suffixes=(
            "_leadershipnow",
            "_openlibrary"
        ),
        validate="one_to_one"
    )
)

In [138]:
matched_book_comparison[
    [
        "book_id",
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "publication_year",
        "first_publish_year",
        "page_count",
        "ratings_average",
        "ratings_count"
    ]
]

,book_id,title_leadershipnow,author,title_openlibrary,authors,publication_year,first_publish_year,page_count,ratings_average,ratings_count
0,BOOK00036,Leadership,Henry Kissinger,Leadership,[Henry Kissinger],2022,2022.0,400.0,NaN,NaN
1,BOOK00857,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,Emotional Intelligence 2.0,"[Travis Bradberry, Jean Greaves, Jean Greaves ...",2022,2009.0,240.0,3.888889,9.0
2,BOOK00866,Emotional Intelligence Habits,Travis Bradberry,Emotional Intelligence Habits,[Travis Bradberry],2023,2023.0,350.0,NaN,NaN
3,BOOK00016,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,The leadership challenge,"[James M. Kouzes, Barry Z. Posner, Elaine Biec...",2023,1987.0,416.0,5.000000,1.0


## Master Dataset Field Integration Strategy

The unified master dataset represents book-level entities while preserving edition-level and source-specific metadata.

Because Open Library and LeadershipNow provide different types of information, field integration does not use a universal source-precedence rule.

Instead:

- Book identity is represented by the project-specific `book_id`.
- Open Library provides work-oriented metadata such as original/first publication year, ratings, reading-interest counts, subjects, descriptions, and multiple edition ISBNs.
- LeadershipNow provides recent release-oriented metadata such as publication date, publication year, format, page count, publisher, cover URL, and source ISBN.
- When both sources represent the same underlying work, complementary metadata is retained rather than overwritten.
- ISBN values are treated as edition identifiers and are not collapsed merely because two source records share a `book_id`.
- Conflicting metadata is preserved or explicitly flagged rather than resolved without evidence.
- Missing information is retained as missing unless another source provides directly compatible information.

This field-level integration strategy separates work identity from edition representation and maintains source provenance throughout the analytical pipeline.

In [140]:
leadershipnow_editions = pd.DataFrame({
    "book_id":
        leadershipnow_canonical["book_id"],

    "scrape_id":
        leadershipnow_canonical["scrape_id"],

    "title":
        leadershipnow_canonical["title"],

    "subtitle":
        leadershipnow_canonical["subtitle"],

    "author":
        leadershipnow_canonical["author"],

    "isbn_source":
        leadershipnow_canonical["isbn"],

    "isbn_normalized":
        leadershipnow_canonical["isbn_normalized"],

    "isbn_13":
        leadershipnow_canonical["isbn_13"],

    "isbn_status":
        leadershipnow_canonical["isbn_status"],

    "publisher":
        leadershipnow_canonical["publisher"],

    "publication_date":
        leadershipnow_canonical["publication_date"],

    "publication_year":
        leadershipnow_canonical["publication_year"],

    "format":
        leadershipnow_canonical["format"],

    "page_count":
        leadershipnow_canonical["page_count"],

    "edition_note":
        leadershipnow_canonical["edition_note"],

    "cover_url":
        leadershipnow_canonical["cover_url"],

    "external_book_url":
        leadershipnow_canonical["external_book_url"],

    "release_pages":
        leadershipnow_canonical["release_pages"],

    "source_metadata_conflict":
        leadershipnow_canonical["source_metadata_conflict"],

    "openlibrary_key":
        leadershipnow_canonical["openlibrary_key"],

    "match_method":
        leadershipnow_canonical["match_method"],

    "match_confidence":
        leadershipnow_canonical["match_confidence"],

    "source":
        "LeadershipNow"
})

In [141]:
print(
    "LeadershipNow edition rows:",
    len(leadershipnow_editions)
)

print(
    "Unique book IDs:",
    leadershipnow_editions[
        "book_id"
    ].nunique()
)

print(
    "Columns:",
    len(leadershipnow_editions.columns)
)

LeadershipNow edition rows: 1120
Unique book IDs: 1120
Columns: 23


In [143]:
shared_ids_check = (
    leadershipnow_editions[
        leadershipnow_editions[
            "book_id"
        ].isin(
            shared_book_ids
        )
    ][
        [
            "book_id",
            "title",
            "author",
            "isbn_13",
            "publication_year",
            "openlibrary_key",
            "match_method"
        ]
    ]
)

shared_ids_check

,book_id,title,author,isbn_13,publication_year,openlibrary_key,match_method
178,BOOK00036,Leadership,Henry Kissinger,9.780593e+12,2022,/works/OL25348188W,Exact normalized title + author
239,BOOK00857,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,9.780975e+12,2022,/works/OL15189300W,Near title >= 90 + author
240,BOOK00866,Emotional Intelligence Habits,Travis Bradberry,9.780975e+12,2023,/works/OL35744456W,Exact normalized title + author
261,BOOK00016,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,9.781120e+12,2023,/works/OL1974749W,Exact normalized title + author


In [144]:
matched_book_comparison = (
    leadershipnow_canonical[
        leadershipnow_canonical[
            "book_id"
        ].isin(
            shared_book_ids
        )
    ][
        [
            "book_id",
            "title",
            "author",
            "publisher",
            "publication_date",
            "publication_year",
            "isbn_13",
            "page_count",
            "format"
        ]
    ]
    .merge(
        openlibrary_integrated[
            openlibrary_integrated[
                "book_id"
            ].isin(
                shared_book_ids
            )
        ][
            [
                "book_id",
                "title",
                "authors",
                "publishers",
                "first_publish_year",
                "isbn_13",
                "description",
                "ratings_average",
                "ratings_count"
            ]
        ],
        on="book_id",
        how="inner",
        suffixes=(
            "_leadershipnow",
            "_openlibrary"
        ),
        validate="one_to_one"
    )
)

In [145]:
matched_book_comparison[
    [
        "book_id",
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "publication_year",
        "first_publish_year",
        "publisher",
        "publishers",
        "format",
        "page_count"
    ]
]

,book_id,title_leadershipnow,author,title_openlibrary,authors,publication_year,first_publish_year,publisher,publishers,format,page_count
0,BOOK00036,Leadership,Henry Kissinger,Leadership,[Henry Kissinger],2022,2022.0,Penguin Press,"[Random House Espanol, Penguin Audio, Penguin ...",Hardcover,400.0
1,BOOK00857,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,Emotional Intelligence 2.0,"[Travis Bradberry, Jean Greaves, Jean Greaves ...",2022,2009.0,TalentSmart,"[TalentSmart, Brilliance Audio, Pgw]",Hardcover,240.0
2,BOOK00866,Emotional Intelligence Habits,Travis Bradberry,Emotional Intelligence Habits,[Travis Bradberry],2023,2023.0,TalentSmart,"[TalentSmart, Incorporated]",Hardcover,350.0
3,BOOK00016,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,The leadership challenge,"[James M. Kouzes, Barry Z. Posner, Elaine Biec...",2023,1987.0,Jossey-Bass,"[Center for Creative Leadership, Jossey Bass W...",Hardcover,416.0


In [146]:
matched_book_comparison[
    [
        "book_id",
        "title_leadershipnow",
        "isbn_13_leadershipnow",
        "isbn_13_openlibrary"
    ]
]

,book_id,title_leadershipnow,isbn_13_leadershipnow,isbn_13_openlibrary
0,BOOK00036,Leadership,9.780593e+12,"[9780141998688, 9780593489451, 9780593489444, ..."
1,BOOK00857,Team Emotional Intelligence 2.0,9.780975e+12,"[9781441842268, 9780974320649, 9781441842237, ..."
2,BOOK00866,Emotional Intelligence Habits,9.780975e+12,[9780974719375]
3,BOOK00016,The Leadership Challenge,9.781120e+12,"[9781119736165, 9781118282489, 9781118606919, ..."


In [147]:
matched_book_comparison[
    [
        "book_id",
        "title_leadershipnow",
        "ratings_average",
        "ratings_count",
        "description"
    ]
]

,book_id,title_leadershipnow,ratings_average,ratings_count,description
0,BOOK00036,Leadership,NaN,NaN,NaN
1,BOOK00857,Team Emotional Intelligence 2.0,3.888889,9.0,In today's fast-paced world of competitive wor...
2,BOOK00866,Emotional Intelligence Habits,NaN,NaN,NaN
3,BOOK00016,The Leadership Challenge,5.000000,1.0,"When it was initially written in 1987, few cou..."


In [151]:
print(
    "LeadershipNow isbn_13 dtype:",
    leadershipnow_canonical[
        "isbn_13"
    ].dtype
)

print(
    "LeadershipNow normalized ISBN dtype:",
    leadershipnow_canonical[
        "isbn_normalized"
    ].dtype
)

print(
    "Edition table ISBN dtype:",
    leadershipnow_editions[
        "isbn_13"
    ].dtype
)

LeadershipNow isbn_13 dtype: float64
LeadershipNow normalized ISBN dtype: str
Edition table ISBN dtype: float64


In [152]:
leadershipnow_canonical[
    "isbn_13"
] = (
    leadershipnow_canonical[
        "isbn_normalized"
    ].astype("string")
)

leadershipnow_editions[
    "isbn_13"
] = (
    leadershipnow_editions[
        "isbn_normalized"
    ].astype("string")
)

In [153]:
leadershipnow_editions[
    [
        "title",
        "isbn_13"
    ]
].head(10)

,title,isbn_13
0,Crisis Capable,9708891880115
1,An Ordinary Man,9780062684165
2,Don't Trust Your Gut,9780062880918
3,Inventor of the Future,9780062947222
4,The Way Forward,9780062994073
5,Jump,9780062999818
6,After Steve,9780063009813
7,The Kingdom of Prep,9780063042643
8,Build,9780063046061
9,Leading with Heart,9780063052932


In [154]:
leadershipnow_editions[
    "isbn_13"
].str.len().value_counts()

isbn_13
13    1117
12       2
14       1
Name: count, dtype: Int64

In [155]:
leadershipnow_editions[
    leadershipnow_editions[
        "book_id"
    ].isin(
        shared_book_ids
    )
][
    [
        "book_id",
        "title",
        "isbn_13",
        "publication_year"
    ]
]

,book_id,title,isbn_13,publication_year
178,BOOK00036,Leadership,9780593489444,2022
239,BOOK00857,Team Emotional Intelligence 2.0,9780974719344,2022
240,BOOK00866,Emotional Intelligence Habits,9780974719375,2023
261,BOOK00016,The Leadership Challenge,9781119736127,2023


In [156]:
openlibrary_books = pd.DataFrame({
    "book_id":
        openlibrary_integrated["book_id"],

    "canonical_title":
        openlibrary_integrated["title"],

    "authors":
        openlibrary_integrated["authors"],

    "description":
        openlibrary_integrated["description"],

    "subjects":
        openlibrary_integrated["work_subjects"],

    "first_publish_year":
        openlibrary_integrated["first_publish_year"],

    "average_rating":
        openlibrary_integrated["ratings_average"],

    "ratings_count":
        openlibrary_integrated["ratings_count"],

    "edition_count":
        openlibrary_integrated["edition_count"],

    "want_to_read_count":
        openlibrary_integrated["want_to_read_count"],

    "currently_reading_count":
        openlibrary_integrated["currently_reading_count"],

    "already_read_count":
        openlibrary_integrated["already_read_count"],

    "cover_url":
        openlibrary_integrated["cover_url"],

    "openlibrary_key":
        openlibrary_integrated["openlibrary_key"],

    "source_openlibrary":
        True,

    "source_leadershipnow":
        openlibrary_integrated[
            "book_id"
        ].isin(shared_book_ids)
})

In [157]:
leadershipnow_only_books = (
    leadershipnow_canonical[
        ~leadershipnow_canonical[
            "book_id"
        ].isin(
            shared_book_ids
        )
    ]
    .copy()
)

In [158]:
print(
    "LeadershipNow-only books:",
    len(leadershipnow_only_books)
)

LeadershipNow-only books: 1116


In [159]:
leadershipnow_books = pd.DataFrame({
    "book_id":
        leadershipnow_only_books["book_id"],

    "canonical_title":
        leadershipnow_only_books["title"],

    "authors":
        leadershipnow_only_books[
            "author"
        ].apply(
            lambda value:
                [value]
                if pd.notna(value)
                else []
        ),

    "description":
        pd.NA,

    "subjects":
        [[] for _ in range(
            len(leadershipnow_only_books)
        )],

    "first_publish_year":
        leadershipnow_only_books[
            "publication_year"
        ],

    "average_rating":
        pd.NA,

    "ratings_count":
        pd.NA,

    "edition_count":
        pd.NA,

    "want_to_read_count":
        pd.NA,

    "currently_reading_count":
        pd.NA,

    "already_read_count":
        pd.NA,

    "cover_url":
        leadershipnow_only_books[
            "cover_url"
        ],

    "openlibrary_key":
        pd.NA,

    "source_openlibrary":
        False,

    "source_leadershipnow":
        True
})

In [160]:
openlibrary_books[
    "publication_year_observed"
] = pd.NA

In [161]:
leadershipnow_books[
    "publication_year_observed"
] = (
    leadershipnow_only_books[
        "publication_year"
    ].values
)

leadershipnow_books[
    "first_publish_year"
] = pd.NA

In [162]:
book_columns = [
    "book_id",
    "canonical_title",
    "authors",
    "description",
    "subjects",
    "first_publish_year",
    "publication_year_observed",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "cover_url",
    "openlibrary_key",
    "source_openlibrary",
    "source_leadershipnow"
]

In [163]:
books = pd.concat(
    [
        openlibrary_books[
            book_columns
        ],
        leadershipnow_books[
            book_columns
        ]
    ],
    ignore_index=True
)

In [164]:
print(
    "Books rows:",
    len(books)
)

print(
    "Unique book IDs:",
    books[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    books[
        "book_id"
    ].duplicated().sum()
)

print(
    "Missing book IDs:",
    books[
        "book_id"
    ].isna().sum()
)

print(
    "Missing canonical titles:",
    books[
        "canonical_title"
    ].isna().sum()
)

Books rows: 2066
Unique book IDs: 2066
Duplicate book IDs: 0
Missing book IDs: 0
Missing canonical titles: 0


In [165]:
source_composition = (
    books.groupby(
        [
            "source_openlibrary",
            "source_leadershipnow"
        ]
    )
    .size()
    .reset_index(
        name="books"
    )
)

source_composition

,source_openlibrary,source_leadershipnow,books
0,False,True,1116
1,True,False,946
2,True,True,4


In [166]:
coverage_summary = pd.DataFrame({
    "field": [
        "canonical_title",
        "authors",
        "description",
        "subjects",
        "first_publish_year",
        "publication_year_observed",
        "average_rating",
        "ratings_count",
        "cover_url"
    ],
    "available": [
        books["canonical_title"].notna().sum(),
        books["authors"].apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        ).sum(),
        books["description"].notna().sum(),
        books["subjects"].apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        ).sum(),
        books["first_publish_year"].notna().sum(),
        books["publication_year_observed"].notna().sum(),
        books["average_rating"].notna().sum(),
        books["ratings_count"].notna().sum(),
        books["cover_url"].notna().sum()
    ]
})

coverage_summary[
    "coverage_pct"
] = (
    coverage_summary[
        "available"
    ]
    /
    len(books)
    *
    100
).round(2)

coverage_summary

,field,available,coverage_pct
0,canonical_title,2066,100.00
1,authors,2058,99.61
2,description,192,9.29
3,subjects,826,39.98
4,first_publish_year,947,45.84
5,publication_year_observed,1116,54.02
6,average_rating,282,13.65
7,ratings_count,282,13.65
8,cover_url,1887,91.34


In [167]:
openlibrary_missing_book_ids = (
    set(openlibrary_integrated["book_id"])
    -
    set(books["book_id"])
)

leadershipnow_missing_book_ids = (
    set(leadershipnow_canonical["book_id"])
    -
    set(books["book_id"])
)

print(
    "Open Library IDs missing from books:",
    len(openlibrary_missing_book_ids)
)

print(
    "LeadershipNow IDs missing from books:",
    len(leadershipnow_missing_book_ids)
)

Open Library IDs missing from books: 0
LeadershipNow IDs missing from books: 0


In [168]:
actual_openlibrary_ids = set(
    openlibrary_integrated["book_id"]
)

actual_leadershipnow_ids = set(
    leadershipnow_canonical["book_id"]
)

books["openlibrary_presence_check"] = (
    books["book_id"].isin(
        actual_openlibrary_ids
    )
)

books["leadershipnow_presence_check"] = (
    books["book_id"].isin(
        actual_leadershipnow_ids
    )
)

openlibrary_flag_errors = (
    books["source_openlibrary"]
    !=
    books["openlibrary_presence_check"]
).sum()

leadershipnow_flag_errors = (
    books["source_leadershipnow"]
    !=
    books["leadershipnow_presence_check"]
).sum()

print(
    "Open Library source flag errors:",
    openlibrary_flag_errors
)

print(
    "LeadershipNow source flag errors:",
    leadershipnow_flag_errors
)

Open Library source flag errors: 0
LeadershipNow source flag errors: 0


In [169]:
books.drop(
    columns=[
        "openlibrary_presence_check",
        "leadershipnow_presence_check"
    ],
    inplace=True
)

In [170]:
book_id_valid = (
    books["book_id"]
    .str.match(
        r"^BOOK\d{5}$"
    )
)

print(
    "Valid book ID format:",
    book_id_valid.sum()
)

print(
    "Invalid book ID format:",
    (~book_id_valid).sum()
)

Valid book ID format: 2066
Invalid book ID format: 0


In [171]:
print(
    "First book ID:",
    books["book_id"].min()
)

print(
    "Last book ID:",
    books["book_id"].max()
)

First book ID: BOOK00001
Last book ID: BOOK02066


In [172]:
cross_source_ids = (
    set(
        openlibrary_integrated[
            "book_id"
        ]
    )
    &
    set(
        leadershipnow_canonical[
            "book_id"
        ]
    )
)

print(
    "Cross-source entities:",
    len(cross_source_ids)
)

books[
    books["book_id"].isin(
        cross_source_ids
    )
][
    [
        "book_id",
        "canonical_title",
        "source_openlibrary",
        "source_leadershipnow"
    ]
]

Cross-source entities: 4


,book_id,canonical_title,source_openlibrary,source_leadershipnow
15,BOOK00016,The leadership challenge,True,True
35,BOOK00036,Leadership,True,True
856,BOOK00857,Emotional Intelligence 2.0,True,True
865,BOOK00866,Emotional Intelligence Habits,True,True


In [173]:
books[
    "title_normalized"
] = (
    books[
        "canonical_title"
    ].apply(
        normalize_title
    )
)

In [174]:
title_frequency = (
    books[
        "title_normalized"
    ]
    .value_counts()
)

print(
    "Unique normalized titles:",
    books[
        "title_normalized"
    ].nunique()
)

print(
    "Normalized titles appearing more than once:",
    (title_frequency > 1).sum()
)

print(
    "Records belonging to repeated-title groups:",
    title_frequency[
        title_frequency > 1
    ].sum()
)

Unique normalized titles: 1718
Normalized titles appearing more than once: 92
Records belonging to repeated-title groups: 440


In [185]:
repeated_title_records = (
    books[
        books[
            "title_normalized"
        ].isin(
            repeated_titles
        )
    ]
    .sort_values(
        "title_normalized"
    )
    [
        [
            "book_id",
            "canonical_title",
            "authors",
            "source_openlibrary",
            "source_leadershipnow"
        ]
    ]
)

repeated_title_records

,book_id,canonical_title,authors,source_openlibrary,source_leadershipnow
308,BOOK00309,A Guide to the Project Management Body of Know...,[Project Management Institute],True,False
558,BOOK00559,A guide to the project management body of know...,[Project Management Institute],True,False
528,BOOK00529,A Guide to the Project Management Body of Know...,[Project Management Institute],True,False
1937,BOOK01938,A New Kind of Diversity,[Tim Elmore],False,True
1998,BOOK01999,A New Kind of Diversity,[Tim Elmore / Foreward by John C. Maxwell],False,True
...,...,...,...,...,...
195,BOOK00196,Transforming leadership,[Leighton Ford],True,False
212,BOOK00213,Transforming leadership,[Terry D. Anderson],True,False
213,BOOK00214,Transforming Leadership,[John D. Adams],True,False
708,BOOK00709,Your Next Five Moves,"[Patrick Bet-David, Greg Dinkin]",True,False


In [186]:
print(
    "Unique normalized titles:",
    books[
        "title_normalized"
    ].nunique()
)

print(
    "Repeated normalized title groups:",
    (
        title_frequency > 1
    ).sum()
)

print(
    "Records in repeated-title groups:",
    len(repeated_title_records)
)

Unique normalized titles: 1718
Repeated normalized title groups: 92
Records in repeated-title groups: 440


In [187]:
FINAL_DIR = Path(
    "../data/final"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Final output directory:",
    FINAL_DIR.resolve()
)

print(
    "Directory exists:",
    FINAL_DIR.exists()
)

Final output directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/final
Directory exists: True


In [188]:
integration_quality = pd.DataFrame({
    "metric": [
        "Raw Open Library records",
        "Canonical Open Library works",
        "Raw LeadershipNow records",
        "Canonical LeadershipNow records",
        "Cross-source matched entities",
        "Final unique book entities",
        "Duplicate book IDs",
        "Missing book IDs",
        "Missing canonical titles"
    ],

    "value": [
        1000,
        len(openlibrary_integrated),
        1124,
        len(leadershipnow_canonical),
        len(cross_source_ids),
        len(books),
        books["book_id"].duplicated().sum(),
        books["book_id"].isna().sum(),
        books["canonical_title"].isna().sum()
    ]
})

integration_quality

,metric,value
0,Raw Open Library records,1000
1,Canonical Open Library works,950
2,Raw LeadershipNow records,1124
3,Canonical LeadershipNow records,1120
4,Cross-source matched entities,4
5,Final unique book entities,2066
6,Duplicate book IDs,0
7,Missing book IDs,0
8,Missing canonical titles,0


In [189]:
FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [190]:
books_export = books.copy()

for column in [
    "authors",
    "subjects"
]:
    books_export[column] = (
        books_export[column].apply(
            lambda x:
                repr(x)
                if isinstance(x, list)
                else x
        )
    )

In [191]:
books_export.to_csv(
    FINAL_DIR /
    "books_master.csv",
    index=False
)

In [192]:
leadershipnow_editions_export = (
    leadershipnow_editions.copy()
)

leadershipnow_editions_export[
    "release_pages"
] = (
    leadershipnow_editions_export[
        "release_pages"
    ].apply(
        lambda x:
            repr(x)
            if isinstance(x, list)
            else x
    )
)

leadershipnow_editions_export.to_csv(
    FINAL_DIR /
    "leadershipnow_editions.csv",
    index=False
)

In [193]:
openlibrary_export = (
    openlibrary_integrated.copy()
)

openlibrary_list_columns = [
    "authors",
    "author_keys",
    "publish_dates",
    "publishers",
    "isbn_10",
    "isbn_13",
    "all_isbns",
    "languages",
    "subjects",
    "collection_queries",
    "work_subjects",
    "subject_places",
    "subject_people",
    "subject_times",
    "excerpts",
    "lc_classifications",
    "dewey_number",
    "work_covers",
    "authors_normalized"
]

for column in openlibrary_list_columns:
    if column in openlibrary_export.columns:
        openlibrary_export[column] = (
            openlibrary_export[column].apply(
                lambda x:
                    repr(x)
                    if isinstance(x, list)
                    else x
            )
        )

openlibrary_export.to_csv(
    FINAL_DIR /
    "openlibrary_integrated.csv",
    index=False
)

In [194]:
integration_quality.to_csv(
    FINAL_DIR /
    "integration_quality_summary.csv",
    index=False
)

coverage_summary.to_csv(
    FINAL_DIR /
    "book_metadata_coverage.csv",
    index=False
)

In [195]:
final_files = [
    "books_master.csv",
    "leadershipnow_editions.csv",
    "openlibrary_integrated.csv",
    "integration_quality_summary.csv",
    "book_metadata_coverage.csv"
]

for filename in final_files:
    filepath = FINAL_DIR / filename

    print(
        filename,
        "->",
        filepath.exists(),
        "|",
        round(
            filepath.stat().st_size / 1024,
            2
        )
        if filepath.exists()
        else 0,
        "KB"
    )

books_master.csv -> True | 640.55 KB
leadershipnow_editions.csv -> True | 377.31 KB
openlibrary_integrated.csv -> True | 1324.71 KB
integration_quality_summary.csv -> True | 0.27 KB
book_metadata_coverage.csv -> True | 0.25 KB


In [197]:
print(
    "books_export:",
    books_export.shape
)

print(
    "leadershipnow_editions_export:",
    leadershipnow_editions_export.shape
)

print(
    "openlibrary_export:",
    openlibrary_export.shape
)

books_export: (2066, 18)
leadershipnow_editions_export: (1120, 23)
openlibrary_export: (950, 50)


In [198]:
books_export.to_csv(
    FINAL_DIR /
    "books_master.csv",
    index=False
)

leadershipnow_editions_export.to_csv(
    FINAL_DIR /
    "leadershipnow_editions.csv",
    index=False
)

openlibrary_export.to_csv(
    FINAL_DIR /
    "openlibrary_integrated.csv",
    index=False
)

integration_quality.to_csv(
    FINAL_DIR /
    "integration_quality_summary.csv",
    index=False
)

coverage_summary.to_csv(
    FINAL_DIR /
    "book_metadata_coverage.csv",
    index=False
)

In [199]:
final_files = [
    "books_master.csv",
    "leadershipnow_editions.csv",
    "openlibrary_integrated.csv",
    "integration_quality_summary.csv",
    "book_metadata_coverage.csv"
]

for filename in final_files:

    filepath = (
        FINAL_DIR /
        filename
    )

    print(
        filename,
        "->",
        filepath.exists(),
        "|",
        round(
            filepath.stat().st_size / 1024,
            2
        )
        if filepath.exists()
        else 0,
        "KB"
    )

books_master.csv -> True | 640.55 KB
leadershipnow_editions.csv -> True | 377.31 KB
openlibrary_integrated.csv -> True | 1324.71 KB
integration_quality_summary.csv -> True | 0.27 KB
book_metadata_coverage.csv -> True | 0.25 KB


In [200]:
books_saved = pd.read_csv(
    FINAL_DIR /
    "books_master.csv"
)

leadershipnow_saved = pd.read_csv(
    FINAL_DIR /
    "leadershipnow_editions.csv"
)

openlibrary_saved = pd.read_csv(
    FINAL_DIR /
    "openlibrary_integrated.csv"
)

In [201]:
print(
    "Books saved:",
    books_saved.shape
)

print(
    "LeadershipNow saved:",
    leadershipnow_saved.shape
)

print(
    "Open Library saved:",
    openlibrary_saved.shape
)

Books saved: (2066, 18)
LeadershipNow saved: (1120, 23)
Open Library saved: (950, 50)


In [202]:
print(
    "Saved books:",
    len(books_saved)
)

print(
    "Unique saved book IDs:",
    books_saved[
        "book_id"
    ].nunique()
)

print(
    "Duplicate saved book IDs:",
    books_saved[
        "book_id"
    ].duplicated().sum()
)

print(
    "Missing saved book IDs:",
    books_saved[
        "book_id"
    ].isna().sum()
)

Saved books: 2066
Unique saved book IDs: 2066
Duplicate saved book IDs: 0
Missing saved book IDs: 0


In [203]:
leadershipnow_saved = pd.read_csv(
    FINAL_DIR /
    "leadershipnow_editions.csv",
    dtype={
        "isbn_13": "string",
        "isbn_normalized": "string",
        "isbn_source": "string"
    }
)

## Conclusion

The integration stage consolidated the two independently collected book sources into a unified entity structure while preserving source provenance and edition-level metadata.

The Open Library collection was consolidated from 1,000 API retrieval records into 950 unique works. LeadershipNow contained 1,124 scraped archive records, which were consolidated into 1,120 canonical source records after resolving four repeated archive ISBN groups.

Cross-source entity resolution identified a small set of records represented in both sources. Rather than deleting source records, matched records were linked through a shared project-specific `book_id`, allowing complementary metadata and provenance to remain available.

The resulting master dataset contains 2,066 unique book entities with no duplicate or missing project identifiers and complete canonical title coverage.

Metadata availability varies by source. Titles and authors have high coverage, while descriptions, subjects, and ratings are concentrated primarily in Open Library. LeadershipNow contributes strong recent publication, edition, ISBN, format, page-count, and cover metadata. Missing values were preserved rather than artificially imputed.

The final integration architecture therefore separates book identity from source and edition representation. This provides a reliable foundation for relational database design, exploratory analysis, NLP feature construction, clustering, and the recommendation system.

In [205]:
[
    name
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
]

['__',
 '___',
 'openlibrary_search',
 'openlibrary_work',
 'leadershipnow',
 'source_key_audit',
 '_5',
 'title_check',
 'openlibrary_work_merge',
 'openlibrary_integrated',
 'work_coverage_df',
 '_15',
 'work_list_coverage_df',
 '_22',
 'openlibrary_isbn_map',
 'ambiguous_isbn_details',
 '_29',
 'openlibrary_isbn_map_safe',
 'leadershipnow_isbn_map',
 'exact_isbn_matches',
 '_41',
 'duplicate_leadershipnow_records',
 '_42',
 'title_match_candidates',
 '_48',
 '_56',
 'high_confidence_matches',
 '_59',
 '_61',
 'leadershipnow_unmatched',
 'openlibrary_title_lookup',
 'strong_title_candidates',
 '_76',
 'fuzzy_candidate_details',
 '_78',
 'fuzzy_author_supported',
 '_82',
 'confirmed_fuzzy_matches',
 '_84',
 'exact_confirmed',
 'fuzzy_confirmed',
 'confirmed_cross_source_matches',
 '_92',
 'matching_summary',
 '_94',
 '_95',
 'duplicate_consistency',
 '_96',
 'page_count_conflicts',
 '_98',
 'conflicting_page_records',
 '_99',
 'archive_provenance',
 '_101',
 'leadershipnow_with_isbn',

In [206]:
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        print(
            name,
            obj.shape,
            list(obj.columns)
        )

__ (4, 5) ['scrape_id', 'title', 'author', 'isbn_normalized', 'publication_year']
___ (4, 2) ['isbn_normalized', 'release_pages']
openlibrary_search (950, 32) ['openlibrary_key', 'title', 'authors', 'author_keys', 'first_publish_year', 'publish_dates', 'publishers', 'isbn_10', 'isbn_13', 'all_isbns', 'languages', 'subjects', 'edition_count', 'ratings_average', 'ratings_count', 'ratings_count_1', 'ratings_count_2', 'ratings_count_3', 'ratings_count_4', 'ratings_count_5', 'want_to_read_count', 'currently_reading_count', 'already_read_count', 'cover_id', 'cover_url', 'ebook_access', 'has_fulltext', 'public_scan', 'source', 'source_url', 'collection_queries', 'query_match_count']
openlibrary_work (950, 13) ['openlibrary_key', 'title', 'work_status_code', 'description', 'first_sentence', 'work_subjects', 'subject_places', 'subject_people', 'subject_times', 'excerpts', 'lc_classifications', 'dewey_number', 'work_covers']
leadershipnow (1124, 28) ['scrape_id', 'title', 'subtitle', 'author', '

In [207]:
confirmed_fuzzy_matches[
    [
        "scrape_id",
        "title_leadershipnow",
        "author",
        "openlibrary_key",
        "title_openlibrary",
        "authors",
        "title_similarity",
        "matched_authors",
        "publication_year",
        "first_publish_year",
        "match_method",
        "match_confidence"
    ]
]

,scrape_id,title_leadershipnow,author,openlibrary_key,title_openlibrary,authors,title_similarity,matched_authors,publication_year,first_publish_year,match_method,match_confidence
1,1012,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,/works/OL15189300W,Emotional Intelligence 2.0,"[Travis Bradberry, Jean Greaves, Jean Greaves ...",91.22807,[jean greaves],2022,2009.0,Near title >= 90 + author,High


In [208]:
confirmed_fuzzy_matches[
    "match_method"
] = pd.Series(dtype="object")

confirmed_fuzzy_matches[
    "match_confidence"
] = pd.Series(dtype="object")

In [209]:
print(
    "Confirmed exact matches:",
    len(high_confidence_matches)
)

print(
    "Confirmed fuzzy matches:",
    len(confirmed_fuzzy_matches)
)

Confirmed exact matches: 3
Confirmed fuzzy matches: 1


In [210]:
# Reject the manually reviewed fuzzy match.
# Keep the same DataFrame structure for downstream compatibility.

confirmed_fuzzy_matches = fuzzy_author_supported.iloc[0:0].copy()

# Add the downstream matching columns to the empty DataFrame.
confirmed_fuzzy_matches["match_method"] = pd.Series(dtype="object")
confirmed_fuzzy_matches["match_confidence"] = pd.Series(dtype="object")

print(
    "Confirmed exact matches:",
    len(high_confidence_matches)
)

print(
    "Confirmed fuzzy matches:",
    len(confirmed_fuzzy_matches)
)

Confirmed exact matches: 3
Confirmed fuzzy matches: 0


In [215]:
# Final confirmed cross-source matches.
# After manual bibliographic review, no fuzzy matches are accepted.

exact_confirmed = high_confidence_matches[
    [
        "scrape_id",
        "openlibrary_key",
        "title_leadershipnow",
        "author",
        "title_openlibrary",
        "authors",
        "publication_year",
        "first_publish_year",
        "match_method",
        "match_confidence"
    ]
].copy()

# No fuzzy matches were accepted after manual review.
fuzzy_confirmed = exact_confirmed.iloc[0:0].copy()

# Final confirmed cross-source matches.
confirmed_cross_source_matches = exact_confirmed.copy()

print(
    "Exact confirmed matches:",
    len(exact_confirmed)
)

print(
    "Fuzzy confirmed matches:",
    len(fuzzy_confirmed)
)

print(
    "Total confirmed cross-source matches:",
    len(confirmed_cross_source_matches)
)

display(
    confirmed_cross_source_matches[
        [
            "title_leadershipnow",
            "author",
            "title_openlibrary",
            "authors",
            "publication_year",
            "first_publish_year",
            "match_method",
            "match_confidence"
        ]
    ]
)

Exact confirmed matches: 3
Fuzzy confirmed matches: 0
Total confirmed cross-source matches: 3


,title_leadershipnow,author,title_openlibrary,authors,publication_year,first_publish_year,match_method,match_confidence
1,Emotional Intelligence Habits,Travis Bradberry,Emotional Intelligence Habits,[Travis Bradberry],2023,2023.0,Exact normalized title + author,High
2,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,The leadership challenge,"[James M. Kouzes, Barry Z. Posner, Elaine Biec...",2023,1987.0,Exact normalized title + author,High
13,Leadership,Henry Kissinger,Leadership,[Henry Kissinger],2022,2022.0,Exact normalized title + author,High


In [216]:
# =====================================================
# REBUILD CROSS-SOURCE MATCH SUMMARY
# =====================================================

matching_summary = pd.DataFrame({
    "matching_stage": [
        "Exact normalized title + author",
        "Accepted fuzzy matches",
        "Total confirmed cross-source matches"
    ],
    "confirmed_matches": [
        len(exact_confirmed),
        len(fuzzy_confirmed),
        len(confirmed_cross_source_matches)
    ]
})

display(matching_summary)

,matching_stage,confirmed_matches
0,Exact normalized title + author,3
1,Accepted fuzzy matches,0
2,Total confirmed cross-source matches,3


In [218]:
# =====================================================
# REBUILD CONFIRMED MATCH MAP
# =====================================================

confirmed_match_map = high_confidence_matches[
    [
        "openlibrary_key",
        "title_leadershipnow",
        "author",
        "title_normalized",
        "author_normalized",
        "match_method",
        "match_confidence"
    ]
].copy()

# Rename normalized fields to the names expected downstream.
confirmed_match_map = confirmed_match_map.rename(
    columns={
        "title_normalized": "leadershipnow_title_normalized",
        "author_normalized": "leadershipnow_author_normalized"
    }
)

print(
    "Confirmed match-map records:",
    len(confirmed_match_map)
)

display(confirmed_match_map)

Confirmed match-map records: 3


,openlibrary_key,title_leadershipnow,author,leadershipnow_title_normalized,leadershipnow_author_normalized,match_method,match_confidence
1,/works/OL35744456W,Emotional Intelligence Habits,Travis Bradberry,emotional intelligence habits,travis bradberry,Exact normalized title + author,High
2,/works/OL1974749W,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,the leadership challenge,james m kouzes and barry z posner,Exact normalized title + author,High
13,/works/OL25348188W,Leadership,Henry Kissinger,leadership,henry kissinger,Exact normalized title + author,High


In [220]:
leadershipnow_canonical.loc[
    leadershipnow_canonical[
        "title"
    ].str.contains(
        "Team Emotional Intelligence 2.0",
        case=False,
        na=False
    ),
    [
        "scrape_id",
        "title",
        "author",
        "publication_year",
        "isbn_normalized",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

,scrape_id,title,author,publication_year,isbn_normalized,openlibrary_key,match_method,match_confidence
239,1012,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,2022,9780974719344,/works/OL15189300W,Near title >= 90 + author,High


In [221]:
# =====================================================
# CLEAN REBUILD OF LEADERSHIPNOW MATCH ASSIGNMENT
# =====================================================

# Start from the canonical 1,120 LeadershipNow records.
leadershipnow_canonical = leadershipnow_canonical_isbn.copy()

# Remove any old downstream matching columns if present.
old_match_columns = [
    "openlibrary_key",
    "leadershipnow_title_normalized",
    "leadershipnow_author_normalized",
    "match_method",
    "match_confidence",
    "book_id"
]

leadershipnow_canonical = leadershipnow_canonical.drop(
    columns=[
        col
        for col in old_match_columns
        if col in leadershipnow_canonical.columns
    ],
    errors="ignore"
)

# Merge ONLY the corrected 3 confirmed matches.
leadershipnow_canonical = leadershipnow_canonical.merge(
    confirmed_match_map,
    left_on=[
        "title_normalized",
        "author_normalized"
    ],
    right_on=[
        "leadershipnow_title_normalized",
        "leadershipnow_author_normalized"
    ],
    how="left",
    validate="many_to_one"
)

# Validate corrected matching.
print(
    "Canonical LeadershipNow records:",
    len(leadershipnow_canonical)
)

print(
    "Matched to Open Library:",
    leadershipnow_canonical["openlibrary_key"].notna().sum()
)

print(
    "Unmatched LeadershipNow:",
    leadershipnow_canonical["openlibrary_key"].isna().sum()
)

Canonical LeadershipNow records: 1120
Matched to Open Library: 3
Unmatched LeadershipNow: 1117


In [223]:
print("Shape:", leadershipnow_canonical.shape)

print("\nAuthor-related columns:")
print([
    col for col in leadershipnow_canonical.columns
    if "author" in col.lower()
])

print("\nMatch-related columns:")
print([
    col for col in leadershipnow_canonical.columns
    if (
        "openlibrary" in col.lower()
        or "match" in col.lower()
    )
])

Shape: (1120, 37)

Author-related columns:
['author_x', 'author_normalized', 'author_y', 'leadershipnow_author_normalized']

Match-related columns:
['openlibrary_key', 'match_method', 'match_confidence']


In [224]:
print(
    "Canonical LeadershipNow records:",
    len(leadershipnow_canonical)
)

print(
    "Matched to Open Library:",
    leadershipnow_canonical["openlibrary_key"].notna().sum()
)

print(
    "Unmatched LeadershipNow:",
    leadershipnow_canonical["openlibrary_key"].isna().sum()
)

Canonical LeadershipNow records: 1120
Matched to Open Library: 3
Unmatched LeadershipNow: 1117


In [226]:
leadershipnow_canonical.loc[
    leadershipnow_canonical["title"].str.contains(
        "Team Emotional Intelligence 2.0",
        case=False,
        na=False
    ),
    [
        "scrape_id",
        "title",
        "publication_year",
        "isbn_normalized",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

,scrape_id,title,publication_year,isbn_normalized,openlibrary_key,match_method,match_confidence
239,1012,Team Emotional Intelligence 2.0,2022,9780974719344,NaN,NaN,NaN


In [227]:
# =====================================================
# CLEAN MERGE ARTIFACT COLUMNS
# =====================================================

leadershipnow_canonical = (
    leadershipnow_canonical
    .rename(columns={"author_x": "author"})
    .drop(columns=["author_y"], errors="ignore")
)

print("Shape:", leadershipnow_canonical.shape)

print("\nAuthor columns:")
print([
    col for col in leadershipnow_canonical.columns
    if "author" in col.lower()
])

Shape: (1120, 36)

Author columns:
['author', 'author_normalized', 'leadershipnow_author_normalized']


In [228]:
# =====================================================
# SPLIT MATCHED AND UNMATCHED LEADERSHIPNOW BOOKS
# =====================================================

leadershipnow_matched = (
    leadershipnow_canonical[
        leadershipnow_canonical["openlibrary_key"].notna()
    ]
    .copy()
)

leadershipnow_unmatched_canonical = (
    leadershipnow_canonical[
        leadershipnow_canonical["openlibrary_key"].isna()
    ]
    .copy()
)

print(
    "Matched LeadershipNow records:",
    len(leadershipnow_matched)
)

print(
    "Unmatched LeadershipNow records:",
    len(leadershipnow_unmatched_canonical)
)

print(
    "Total:",
    len(leadershipnow_matched)
    + len(leadershipnow_unmatched_canonical)
)

Matched LeadershipNow records: 3
Unmatched LeadershipNow records: 1117
Total: 1120


In [229]:
display(
    leadershipnow_matched[
        [
            "title",
            "author",
            "publication_year",
            "openlibrary_key",
            "match_method",
            "match_confidence"
        ]
    ]
)

,title,author,publication_year,openlibrary_key,match_method,match_confidence
178,Leadership,Henry Kissinger,2022,/works/OL25348188W,Exact normalized title + author,High
240,Emotional Intelligence Habits,Travis Bradberry,2023,/works/OL35744456W,Exact normalized title + author,High
261,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,2023,/works/OL1974749W,Exact normalized title + author,High


In [230]:
print(
    "Open Library master shape:",
    openlibrary_master.shape
)

print(
    "Unique book IDs:",
    openlibrary_master["book_id"].nunique()
)

print(
    "Missing book IDs:",
    openlibrary_master["book_id"].isna().sum()
)

display(
    openlibrary_master[
        [
            "book_id",
            "title",
            "openlibrary_key"
        ]
    ].head(10)
)

Open Library master shape: (950, 23)
Unique book IDs: 950
Missing book IDs: 0


,book_id,title,openlibrary_key
0,BOOK00001,Principle-Centered Leadership,/works/OL2630041W
1,BOOK00002,Leadership in Organizations,/works/OL2731767W
2,BOOK00003,Kepemimpinan =,/works/OL302757W
3,BOOK00004,Spiritual leadership,/works/OL450702W
4,BOOK00005,Leadership,/works/OL94176W
5,BOOK00006,The 21 Irrefutable Laws of Leadership,/works/OL28124W
6,BOOK00007,Leadership,/works/OL7956784W
7,BOOK00008,Leadership and performance beyond expectations,/works/OL2421691W
8,BOOK00009,A Higher Loyalty,/works/OL18146967W
9,BOOK00010,Leadership and Self Deception,/works/OL8921343W


In [231]:
# =====================================================
# MAP CONFIRMED MATCHES TO OPEN LIBRARY BOOK IDs
# =====================================================

openlibrary_book_id_map = (
    openlibrary_master[
        [
            "openlibrary_key",
            "book_id"
        ]
    ]
    .drop_duplicates()
)

leadershipnow_matched = (
    leadershipnow_matched
    .drop(columns=["book_id"], errors="ignore")
    .merge(
        openlibrary_book_id_map,
        on="openlibrary_key",
        how="left",
        validate="many_to_one"
    )
)

print(
    "Matched LeadershipNow records:",
    len(leadershipnow_matched)
)

print(
    "Missing inherited book IDs:",
    leadershipnow_matched["book_id"].isna().sum()
)

print(
    "Unique inherited book IDs:",
    leadershipnow_matched["book_id"].nunique()
)

display(
    leadershipnow_matched[
        [
            "book_id",
            "title",
            "author",
            "openlibrary_key",
            "match_method"
        ]
    ]
)

Matched LeadershipNow records: 3
Missing inherited book IDs: 0
Unique inherited book IDs: 3


,book_id,title,author,openlibrary_key,match_method
0,BOOK00036,Leadership,Henry Kissinger,/works/OL25348188W,Exact normalized title + author
1,BOOK00866,Emotional Intelligence Habits,Travis Bradberry,/works/OL35744456W,Exact normalized title + author
2,BOOK00016,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,/works/OL1974749W,Exact normalized title + author


In [232]:
# =====================================================
# ASSIGN BOOK IDs TO LEADERSHIPNOW-ONLY BOOKS
# =====================================================

leadershipnow_unmatched_canonical = (
    leadershipnow_unmatched_canonical
    .drop(columns=["book_id"], errors="ignore")
    .copy()
)

start_book_number = (
    openlibrary_master["book_id"]
    .str.replace("BOOK", "", regex=False)
    .astype(int)
    .max()
    + 1
)

leadershipnow_unmatched_canonical["book_id"] = [
    f"BOOK{i:05d}"
    for i in range(
        start_book_number,
        start_book_number + len(leadershipnow_unmatched_canonical)
    )
]

print("Starting new book ID:", f"BOOK{start_book_number:05d}")

print(
    "LeadershipNow-only books:",
    len(leadershipnow_unmatched_canonical)
)

print(
    "Unique new book IDs:",
    leadershipnow_unmatched_canonical["book_id"].nunique()
)

print(
    "Missing book IDs:",
    leadershipnow_unmatched_canonical["book_id"].isna().sum()
)

print(
    "First new book ID:",
    leadershipnow_unmatched_canonical["book_id"].iloc[0]
)

print(
    "Last new book ID:",
    leadershipnow_unmatched_canonical["book_id"].iloc[-1]
)

Starting new book ID: BOOK00951
LeadershipNow-only books: 1117
Unique new book IDs: 1117
Missing book IDs: 0
First new book ID: BOOK00951
Last new book ID: BOOK02067


In [233]:
leadershipnow_unmatched_canonical.loc[
    leadershipnow_unmatched_canonical["title"].str.contains(
        "Team Emotional Intelligence 2.0",
        case=False,
        na=False
    ),
    [
        "book_id",
        "title",
        "author",
        "publication_year",
        "isbn_normalized",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

,book_id,title,author,publication_year,isbn_normalized,openlibrary_key,match_method,match_confidence
239,BOOK01189,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,2022,9780974719344,NaN,NaN,NaN


In [234]:
openlibrary_ids = set(
    openlibrary_master["book_id"]
)

leadershipnow_only_ids = set(
    leadershipnow_unmatched_canonical["book_id"]
)

shared_ids = (
    openlibrary_ids
    & leadershipnow_only_ids
)

print(
    "Open Library IDs:",
    len(openlibrary_ids)
)

print(
    "LeadershipNow-only IDs:",
    len(leadershipnow_only_ids)
)

print(
    "ID collisions:",
    len(shared_ids)
)

print(
    "Expected unique master books:",
    len(openlibrary_ids)
    + len(leadershipnow_only_ids)
)

Open Library IDs: 950
LeadershipNow-only IDs: 1117
ID collisions: 0
Expected unique master books: 2067


In [235]:
# =====================================================
# BUILD LEADERSHIPNOW-ONLY MASTER BOOK RECORDS
# =====================================================

leadershipnow_books = pd.DataFrame({
    "book_id": leadershipnow_unmatched_canonical["book_id"],
    "canonical_title": leadershipnow_unmatched_canonical["title"],
    "authors": leadershipnow_unmatched_canonical["author"].apply(
        lambda x: [x] if pd.notna(x) else []
    ),
    "description": None,
    "subjects": [[] for _ in range(len(leadershipnow_unmatched_canonical))],
    "first_publish_year": np.nan,
    "average_rating": np.nan,
    "ratings_count": np.nan,
    "edition_count": np.nan,
    "want_to_read_count": np.nan,
    "currently_reading_count": np.nan,
    "already_read_count": np.nan,
    "cover_url": leadershipnow_unmatched_canonical["cover_url"],
    "openlibrary_key": None,
    "source_openlibrary": False,
    "source_leadershipnow": True,
    "publication_year_observed": leadershipnow_unmatched_canonical[
        "publication_year"
    ]
})

print(
    "LeadershipNow-only master records:",
    len(leadershipnow_books)
)

print(
    "Unique book IDs:",
    leadershipnow_books["book_id"].nunique()
)

print(
    "Missing book IDs:",
    leadershipnow_books["book_id"].isna().sum()
)

display(leadershipnow_books.head())

LeadershipNow-only master records: 1117
Unique book IDs: 1117
Missing book IDs: 0


,book_id,canonical_title,authors,description,subjects,first_publish_year,average_rating,ratings_count,edition_count,want_to_read_count,currently_reading_count,already_read_count,cover_url,openlibrary_key,source_openlibrary,source_leadershipnow,publication_year_observed
0,BOOK00951,Crisis Capable,[Fabiana Lacerca-Allen],None,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,None,False,True,2024
1,BOOK00952,An Ordinary Man,[Richard Norton Smith],None,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,None,False,True,2023
2,BOOK00953,Don't Trust Your Gut,[Seth Stephens-Davidowitz],None,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,None,False,True,2022
3,BOOK00954,Inventor of the Future,[Alec Nevala-Lee],None,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,None,False,True,2022
4,BOOK00955,The Way Forward,[Robert O'Neill and Dakota Meyer],None,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,None,False,True,2022


In [236]:
# =====================================================
# REBUILD OPEN LIBRARY MASTER RECORDS
# =====================================================

confirmed_openlibrary_keys = set(
    confirmed_cross_source_matches["openlibrary_key"]
)

openlibrary_books = pd.DataFrame({
    "book_id": openlibrary_master["book_id"],
    "canonical_title": openlibrary_master["title"],
    "authors": openlibrary_master["authors"],
    "description": openlibrary_master["description"],
    "subjects": openlibrary_master["categories"],
    "first_publish_year": openlibrary_master["first_publish_year"],
    "average_rating": openlibrary_master["average_rating"],
    "ratings_count": openlibrary_master["ratings_count"],
    "edition_count": openlibrary_master["edition_count"],
    "want_to_read_count": openlibrary_master["want_to_read_count"],
    "currently_reading_count": openlibrary_master["currently_reading_count"],
    "already_read_count": openlibrary_master["already_read_count"],
    "cover_url": openlibrary_master["cover_url"],
    "openlibrary_key": openlibrary_master["openlibrary_key"],
    "source_openlibrary": True,
    "source_leadershipnow": openlibrary_master[
        "openlibrary_key"
    ].isin(confirmed_openlibrary_keys),
    "publication_year_observed": np.nan
})

print("Open Library master records:", len(openlibrary_books))

print(
    "Open Library records also found in LeadershipNow:",
    openlibrary_books["source_leadershipnow"].sum()
)

Open Library master records: 950
Open Library records also found in LeadershipNow: 3


In [237]:
# =====================================================
# BUILD CORRECTED MASTER BOOK DATASET
# =====================================================

books = pd.concat(
    [
        openlibrary_books,
        leadershipnow_books
    ],
    ignore_index=True
)

books["title_normalized"] = (
    books["canonical_title"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Master book records:", len(books))
print("Unique book IDs:", books["book_id"].nunique())
print("Duplicate book IDs:", books["book_id"].duplicated().sum())
print("Missing book IDs:", books["book_id"].isna().sum())

Master book records: 2067
Unique book IDs: 2067
Duplicate book IDs: 0
Missing book IDs: 0


In [238]:
source_composition = (
    books
    .groupby(
        [
            "source_openlibrary",
            "source_leadershipnow"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="books")
)

display(source_composition)

,source_openlibrary,source_leadershipnow,books
0,False,True,1117
1,True,False,947
2,True,True,3


In [239]:
# =====================================================
# BUILD CORRECTED LEADERSHIPNOW BOOK-ID MAP
# =====================================================

matched_id_map = leadershipnow_matched[
    [
        "scrape_id",
        "book_id"
    ]
].copy()

unmatched_id_map = leadershipnow_unmatched_canonical[
    [
        "scrape_id",
        "book_id"
    ]
].copy()

leadershipnow_book_id_map = pd.concat(
    [
        matched_id_map,
        unmatched_id_map
    ],
    ignore_index=True
)

print(
    "LeadershipNow ID-map records:",
    len(leadershipnow_book_id_map)
)

print(
    "Unique scrape IDs:",
    leadershipnow_book_id_map["scrape_id"].nunique()
)

print(
    "Missing book IDs:",
    leadershipnow_book_id_map["book_id"].isna().sum()
)

print(
    "Duplicate scrape IDs:",
    leadershipnow_book_id_map["scrape_id"].duplicated().sum()
)

LeadershipNow ID-map records: 1120
Unique scrape IDs: 1120
Missing book IDs: 0
Duplicate scrape IDs: 0


In [240]:
# =====================================================
# REBUILD LEADERSHIPNOW EDITION BOOK IDs
# =====================================================

leadershipnow_editions = (
    leadershipnow_editions
    .drop(columns=["book_id"], errors="ignore")
    .merge(
        leadershipnow_book_id_map,
        on="scrape_id",
        how="left",
        validate="one_to_one"
    )
)

# Put book_id first again.
edition_columns = (
    ["book_id"]
    + [
        col
        for col in leadershipnow_editions.columns
        if col != "book_id"
    ]
)

leadershipnow_editions = leadershipnow_editions[
    edition_columns
]

print(
    "LeadershipNow editions:",
    len(leadershipnow_editions)
)

print(
    "Unique scrape IDs:",
    leadershipnow_editions["scrape_id"].nunique()
)

print(
    "Missing book IDs:",
    leadershipnow_editions["book_id"].isna().sum()
)

print(
    "Invalid book IDs:",
    (~leadershipnow_editions["book_id"].isin(
        books["book_id"]
    )).sum()
)

LeadershipNow editions: 1120
Unique scrape IDs: 1120
Missing book IDs: 0
Invalid book IDs: 0


In [241]:
leadershipnow_editions.loc[
    leadershipnow_editions["title"].str.contains(
        "Team Emotional Intelligence 2.0",
        case=False,
        na=False
    ),
    [
        "book_id",
        "scrape_id",
        "title",
        "author",
        "isbn_13",
        "publication_year",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

,book_id,scrape_id,title,author,isbn_13,publication_year,openlibrary_key,match_method,match_confidence
239,BOOK01189,1012,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,9780974719344,2022,/works/OL15189300W,Near title >= 90 + author,High


In [242]:
leadershipnow_editions.loc[
    leadershipnow_editions["openlibrary_key"].notna(),
    [
        "book_id",
        "title",
        "author",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

,book_id,title,author,openlibrary_key,match_method,match_confidence
178,BOOK00036,Leadership,Henry Kissinger,/works/OL25348188W,Exact normalized title + author,High
239,BOOK01189,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,/works/OL15189300W,Near title >= 90 + author,High
240,BOOK00866,Emotional Intelligence Habits,Travis Bradberry,/works/OL35744456W,Exact normalized title + author,High
261,BOOK00016,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,/works/OL1974749W,Exact normalized title + author,High


In [243]:
# =====================================================
# REBUILD EDITION MATCH METADATA
# =====================================================

# Correct match metadata from the rebuilt canonical dataset.
corrected_match_metadata = leadershipnow_canonical[
    [
        "scrape_id",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
].copy()

# Remove stale match metadata from the old edition table.
leadershipnow_editions = leadershipnow_editions.drop(
    columns=[
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ],
    errors="ignore"
)

# Attach the corrected metadata.
leadershipnow_editions = leadershipnow_editions.merge(
    corrected_match_metadata,
    on="scrape_id",
    how="left",
    validate="one_to_one"
)

print(
    "LeadershipNow editions:",
    len(leadershipnow_editions)
)

print(
    "Editions matched to Open Library:",
    leadershipnow_editions["openlibrary_key"].notna().sum()
)

print(
    "Editions without Open Library match:",
    leadershipnow_editions["openlibrary_key"].isna().sum()
)

LeadershipNow editions: 1120
Editions matched to Open Library: 3
Editions without Open Library match: 1117


In [244]:
leadershipnow_editions.loc[
    leadershipnow_editions["title"].str.contains(
        "Team Emotional Intelligence 2.0",
        case=False,
        na=False
    ),
    [
        "book_id",
        "scrape_id",
        "title",
        "author",
        "isbn_13",
        "publication_year",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

,book_id,scrape_id,title,author,isbn_13,publication_year,openlibrary_key,match_method,match_confidence
239,BOOK01189,1012,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,9780974719344,2022,NaN,NaN,NaN


In [245]:
shared_edition_check = leadershipnow_editions.loc[
    leadershipnow_editions["openlibrary_key"].notna(),
    [
        "book_id",
        "title",
        "author",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
].copy()

print(
    "Confirmed shared editions:",
    len(shared_edition_check)
)

display(shared_edition_check)

Confirmed shared editions: 3


,book_id,title,author,openlibrary_key,match_method,match_confidence
178,BOOK00036,Leadership,Henry Kissinger,/works/OL25348188W,Exact normalized title + author,High
240,BOOK00866,Emotional Intelligence Habits,Travis Bradberry,/works/OL35744456W,Exact normalized title + author,High
261,BOOK00016,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,/works/OL1974749W,Exact normalized title + author,High


In [246]:
print(
    "Master books:",
    len(books)
)

print(
    "Unique master book IDs:",
    books["book_id"].nunique()
)

print(
    "LeadershipNow editions:",
    len(leadershipnow_editions)
)

print(
    "Unique edition scrape IDs:",
    leadershipnow_editions["scrape_id"].nunique()
)

print(
    "Missing edition book IDs:",
    leadershipnow_editions["book_id"].isna().sum()
)

print(
    "Invalid edition → book relationships:",
    (~leadershipnow_editions["book_id"].isin(
        books["book_id"]
    )).sum()
)

print(
    "Confirmed cross-source entities:",
    len(confirmed_cross_source_matches)
)

Master books: 2067
Unique master book IDs: 2067
LeadershipNow editions: 1120
Unique edition scrape IDs: 1120
Missing edition book IDs: 0
Invalid edition → book relationships: 0
Confirmed cross-source entities: 3


In [247]:
# =====================================================
# REBUILD INTEGRATION QUALITY SUMMARY
# =====================================================

integration_quality = pd.DataFrame({
    "metric": [
        "Open Library books",
        "LeadershipNow canonical books",
        "Confirmed cross-source matches",
        "Open Library-only books",
        "LeadershipNow-only books",
        "Both-source books",
        "Final unique book entities",
        "Duplicate master book IDs",
        "Invalid edition-book relationships"
    ],
    "value": [
        len(openlibrary_books),
        len(leadershipnow_editions),
        len(confirmed_cross_source_matches),
        ((books["source_openlibrary"]) &
         (~books["source_leadershipnow"])).sum(),
        ((~books["source_openlibrary"]) &
         (books["source_leadershipnow"])).sum(),
        ((books["source_openlibrary"]) &
         (books["source_leadershipnow"])).sum(),
        len(books),
        books["book_id"].duplicated().sum(),
        (~leadershipnow_editions["book_id"].isin(
            books["book_id"]
        )).sum()
    ]
})

display(integration_quality)

,metric,value
0,Open Library books,950
1,LeadershipNow canonical books,1120
2,Confirmed cross-source matches,3
3,Open Library-only books,947
4,LeadershipNow-only books,1117
5,Both-source books,3
6,Final unique book entities,2067
7,Duplicate master book IDs,0
8,Invalid edition-book relationships,0


In [248]:
# =====================================================
# REBUILD MASTER METADATA COVERAGE
# =====================================================

coverage_fields = [
    "canonical_title",
    "authors",
    "description",
    "subjects",
    "first_publish_year",
    "publication_year_observed",
    "average_rating",
    "ratings_count",
    "cover_url"
]

coverage_rows = []

for field in coverage_fields:

    if field in ["authors", "subjects"]:

        available = books[field].apply(
            lambda x: isinstance(x, list) and len(x) > 0
        ).sum()

    else:

        available = books[field].notna().sum()

    coverage_rows.append({
        "field": field,
        "available": int(available),
        "coverage_pct": round(
            available / len(books) * 100,
            2
        )
    })

coverage_summary = pd.DataFrame(coverage_rows)

display(coverage_summary)

,field,available,coverage_pct
0,canonical_title,2067,100.00
1,authors,2059,99.61
2,description,192,9.29
3,subjects,826,39.96
4,first_publish_year,947,45.82
5,publication_year_observed,1117,54.04
6,average_rating,282,13.64
7,ratings_count,282,13.64
8,cover_url,1888,91.34


In [249]:
# =====================================================
# PREPARE CORRECTED FINAL EXPORTS
# =====================================================

books_export = books.copy()

leadershipnow_editions_export = (
    leadershipnow_editions.copy()
)

openlibrary_export = (
    openlibrary_integrated.copy()
)

print("books_export:", books_export.shape)
print(
    "leadershipnow_editions_export:",
    leadershipnow_editions_export.shape
)
print(
    "openlibrary_export:",
    openlibrary_export.shape
)

books_export: (2067, 18)
leadershipnow_editions_export: (1120, 23)
openlibrary_export: (950, 50)


In [250]:
# =====================================================
# FINAL PRE-SAVE VALIDATION
# =====================================================

assert len(books_export) == 2067
assert books_export["book_id"].nunique() == 2067
assert books_export["book_id"].isna().sum() == 0
assert books_export["book_id"].duplicated().sum() == 0

assert len(leadershipnow_editions_export) == 1120
assert leadershipnow_editions_export["scrape_id"].nunique() == 1120
assert leadershipnow_editions_export["book_id"].isna().sum() == 0

assert (
    leadershipnow_editions_export[
        "openlibrary_key"
    ].notna().sum()
    == 3
)

assert (
    ~leadershipnow_editions_export["book_id"].isin(
        books_export["book_id"]
    )
).sum() == 0

assert len(confirmed_cross_source_matches) == 3

print("All corrected integration assertions passed.")

All corrected integration assertions passed.


In [251]:
# =====================================================
# SAVE CORRECTED FINAL INTEGRATION FILES
# =====================================================

from pathlib import Path

final_dir = Path("../data/final")
final_dir.mkdir(parents=True, exist_ok=True)

books_path = final_dir / "books_master.csv"
editions_path = final_dir / "leadershipnow_editions.csv"
openlibrary_path = final_dir / "openlibrary_integrated.csv"
quality_path = final_dir / "integration_quality_summary.csv"
coverage_path = final_dir / "book_metadata_coverage.csv"

books_export.to_csv(
    books_path,
    index=False
)

leadershipnow_editions_export.to_csv(
    editions_path,
    index=False
)

openlibrary_export.to_csv(
    openlibrary_path,
    index=False
)

integration_quality.to_csv(
    quality_path,
    index=False
)

coverage_summary.to_csv(
    coverage_path,
    index=False
)

print("Corrected final files saved:")
print(books_path)
print(editions_path)
print(openlibrary_path)
print(quality_path)
print(coverage_path)

Corrected final files saved:
../data/final/books_master.csv
../data/final/leadershipnow_editions.csv
../data/final/openlibrary_integrated.csv
../data/final/integration_quality_summary.csv
../data/final/book_metadata_coverage.csv


In [252]:
# =====================================================
# RELOAD CORRECTED FILES FROM DISK
# =====================================================

books_check = pd.read_csv(books_path)

editions_check = pd.read_csv(editions_path)

openlibrary_check = pd.read_csv(openlibrary_path)

quality_check = pd.read_csv(quality_path)

coverage_check = pd.read_csv(coverage_path)

print("Reloaded file shapes:")
print("books_master:", books_check.shape)
print("leadershipnow_editions:", editions_check.shape)
print("openlibrary_integrated:", openlibrary_check.shape)
print("integration_quality_summary:", quality_check.shape)
print("book_metadata_coverage:", coverage_check.shape)

Reloaded file shapes:
books_master: (2067, 18)
leadershipnow_editions: (1120, 23)
openlibrary_integrated: (950, 50)
integration_quality_summary: (9, 2)
book_metadata_coverage: (9, 3)


In [253]:
# =====================================================
# VALIDATE FILES AFTER RELOADING
# =====================================================

assert len(books_check) == 2067
assert books_check["book_id"].nunique() == 2067
assert books_check["book_id"].isna().sum() == 0
assert books_check["book_id"].duplicated().sum() == 0

assert len(editions_check) == 1120
assert editions_check["scrape_id"].nunique() == 1120
assert editions_check["book_id"].isna().sum() == 0

assert editions_check["openlibrary_key"].notna().sum() == 3

assert (
    ~editions_check["book_id"].isin(
        books_check["book_id"]
    )
).sum() == 0

print("Saved-file validation passed.")

Saved-file validation passed.


In [254]:
team_ei_check = editions_check.loc[
    editions_check["title"].str.contains(
        "Team Emotional Intelligence 2.0",
        case=False,
        na=False
    ),
    [
        "book_id",
        "title",
        "author",
        "isbn_13",
        "publication_year",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

display(team_ei_check)

,book_id,title,author,isbn_13,publication_year,openlibrary_key,match_method,match_confidence
239,BOOK01189,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,9780974719344,2022,NaN,NaN,NaN


In [255]:
saved_shared_check = editions_check.loc[
    editions_check["openlibrary_key"].notna(),
    [
        "book_id",
        "title",
        "author",
        "openlibrary_key",
        "match_method",
        "match_confidence"
    ]
]

print(
    "Confirmed shared records on disk:",
    len(saved_shared_check)
)

display(saved_shared_check)

Confirmed shared records on disk: 3


,book_id,title,author,openlibrary_key,match_method,match_confidence
178,BOOK00036,Leadership,Henry Kissinger,/works/OL25348188W,Exact normalized title + author,High
240,BOOK00866,Emotional Intelligence Habits,Travis Bradberry,/works/OL35744456W,Exact normalized title + author,High
261,BOOK00016,The Leadership Challenge,James M. Kouzes and Barry Z. Posner,/works/OL1974749W,Exact normalized title + author,High


## Conclusion

The integration stage consolidated the two independently collected sources into a unified book-entity structure while preserving source provenance and edition-level metadata.

The Open Library collection contains **950 unique works** after consolidating the original 1,000 API retrieval records.

LeadershipNow contains **1,120 canonical records** after consolidating the 1,124 scraped archive records and resolving four repeated ISBN groups.

Cross-source entity resolution confirmed **3 books represented in both sources**. Fuzzy title matching was used only for candidate identification; no fuzzy candidate was ultimately accepted as a confirmed match.

The final integrated catalogue therefore contains:

- **947 Open Library-only books**
- **1,117 LeadershipNow-only books**
- **3 books represented in both sources**
- **2,067 unique canonical book entities**

Each canonical book has a unique project-specific `book_id`, while source and edition information remains separately traceable.

The resulting data architecture provides a validated foundation for relational database design, exploratory analysis, statistical analysis, NLP, feature engineering, clustering, and recommendation modelling.